In [24]:
import math
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
import statsmodels.formula.api as smf
import torch
from IPython.display import display
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# Run from `code/` and expose both `code/` and `code/kev/` on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from data.base import (
    build_long_records,
    fused_eval_context,
    group_records_by_state,
    make_grouped_view,
    make_view,
    predict_probs_grouped,
)
from data.berke2023 import load_berke2023_df
from data.berke2023.stimulus import CHOICE_OPTIONS as BERKE_CHOICE_OPTIONS
from data.covid import (
    AMCE_FORMULA as COVID_AMCE_FORMULA,
    AMCE_TERMS as COVID_AMCE_TERMS,
    _profile_attr_dict as covid_profile_attr_dict,
    load_covid_df,
)
from data.grummon2023 import GRUMMON2023_PERSONA_COLS, load_grummon2023_df
from data.grummon2025 import (
    GRUMMON2025_PERSONA_COLS,
    PRIMARY_CHOICE_STEMS as GRUMMON2025_PRIMARY_STEMS,
    TREAT_ARMS as GRUMMON2025_TREAT_ARMS,
    evaluate_grummon2025_ols,
    load_grummon2025_df,
)
from data.Kriner2026 import (
    AMCE_FORMULA as KRINER_AMCE_FORMULA,
    AMCE_TERMS as KRINER_AMCE_TERMS,
    load_kriner2026_df,
    parse_flight_description,
)
from data.Kriner2026.stimulus import CHOICE_OPTIONS as KRINER_CHOICE_OPTIONS
from data.nutrition.grummon2023.stimulus import (
    ARM_NAMES as GRUMMON2023_ARM_NAMES,
    build_product_catalog as build_grummon2023_product_catalog,
    load_analytic_cohort as load_grummon2023_analytic_cohort,
)
from data.wu2025 import load_wu2025_df
from data.wu2025.stimulus import ATTR_LOCAL, ATTR_MONEY, ATTR_PERCENT, CONJOINT_OPTIONS as WU_CONJOINT_OPTIONS
from models.kev_pointer import KevPointerModel
from training import free_gpu

print(f"CODE_DIR: {CODE_DIR}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code


# Dataset 1: Wu 2025 (electricity-plan conjoint — future train)

In [2]:
# Load Wu 2025 survey rows and filter to paired electricity-plan conjoint choices (`plan_choice`)
t_wu0 = time.perf_counter()
wu2025_df = load_wu2025_df()
wu2025_choice_df = wu2025_df[wu2025_df["v"] == "plan_choice"].reset_index(drop=True)
wu2025_load_sec = time.perf_counter() - t_wu0
if len(wu2025_choice_df) == 0:
    raise ValueError("No `plan_choice` rows found in wu2025_df")

display(
    pd.Series(
        {
            "total_exported_rows": len(wu2025_df),
            "exported_questions": wu2025_df["v"].nunique(),
            "plan_choice_tasks": len(wu2025_choice_df),
            "plan_choice_option_rows": len(wu2025_choice_df) * 2,
            "respondents": wu2025_choice_df["CASEID"].nunique(),
            "options_per_task": 2,
            "load_sec": round(wu2025_load_sec, 4),
        },
        name="wu2025_overview",
    ).to_frame()
)

,wu2025_overview
total_exported_rows,90979.0000
exported_questions,46.0000
plan_choice_tasks,18591.0000
plan_choice_option_rows,37182.0000
respondents,1861.0000
options_per_task,2.0000
load_sec,0.2032


In [3]:
# Parse each displayed electricity plan into its 3 randomized attributes (`money`, `percent`, `local`)
WU2025_LEVELS = {
    ATTR_MONEY: ["$105", "$110", "$115", "$120", "$125"],
    ATTR_PERCENT: ["10%", "30%", "50%", "100%"],
    ATTR_LOCAL: ["No", "Yes"],
}

wu_opt_rows: list[dict] = []
for t_idx, row in enumerate(wu2025_choice_df.itertuples(index=False)):
    if list(row.option_texts) != WU_CONJOINT_OPTIONS:
        raise ValueError(f"Wu 2025 task {t_idx}: expected {WU_CONJOINT_OPTIONS}, got {row.option_texts}")
    if row.option_descriptions is None or len(row.option_descriptions) != 2:
        raise ValueError(f"Wu 2025 task {t_idx}: expected 2 option_descriptions")
    for desc, y_val in zip(row.option_descriptions, row.one_hot, strict=True):
        kv = dict(clause.split(": ", 1) for clause in desc.split("; "))
        for attr, valid_levels in WU2025_LEVELS.items():
            if kv[attr] not in valid_levels:
                raise ValueError(f"Wu 2025 task {t_idx}: invalid {attr}={kv[attr]!r}")
        wu_opt_rows.append(
            {
                "CASEID": row.CASEID,
                "money": kv[ATTR_MONEY],
                "percent": kv[ATTR_PERCENT],
                "local": kv[ATTR_LOCAL],
                "y": float(y_val),
            }
        )

wu2025_opt_df = pd.DataFrame(wu_opt_rows)
WU2025_SIGMA_Y = float(np.std(wu2025_opt_df["y"].to_numpy(), ddof=0))
if WU2025_SIGMA_Y <= 0.0:
    raise ValueError(f"Non-positive WU2025_SIGMA_Y: {WU2025_SIGMA_Y}")

# Fit respondent-clustered human AMCE OLS (`$105`, `10%`, and `No` as reference levels)
wu_fit = smf.ols(
    "y ~ C(money, Treatment(reference='$105')) "
    "+ C(percent, Treatment(reference='10%')) "
    "+ C(local, Treatment(reference='No'))",
    data=wu2025_opt_df,
).fit(cov_type="cluster", cov_kwds={"groups": wu2025_opt_df["CASEID"]})

wu_term_specs = [
    *(
        (f"C(money, Treatment(reference='$105'))[T.{lvl}]", f"{ATTR_MONEY}: {lvl} vs $105")
        for lvl in WU2025_LEVELS[ATTR_MONEY][1:]
    ),
    *(
        (f"C(percent, Treatment(reference='10%'))[T.{lvl}]", f"{ATTR_PERCENT}: {lvl} vs 10%")
        for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]
    ),
    (f"C(local, Treatment(reference='No'))[T.Yes]", f"{ATTR_LOCAL}: Yes vs No"),
]

wu2025_human_ols_df = pd.DataFrame(
    [
        {
            "term": label,
            "human_beta": float(wu_fit.params[p_term]),
            "cluster_se": float(wu_fit.bse[p_term]),
            "p_value": float(wu_fit.pvalues[p_term]),
            "sigma_y": WU2025_SIGMA_Y,
            "std_beta": float(wu_fit.params[p_term]) / WU2025_SIGMA_Y,
        }
        for p_term, label in wu_term_specs
    ]
).set_index("term")

display(wu2025_human_ols_df.round(4))

,human_beta,cluster_se,p_value,sigma_y,std_beta
term,,,,,
Monthly bill: $110 vs $105,-0.0969,0.0078,0.0,0.5,-0.1938
Monthly bill: $115 vs $105,-0.1779,0.0084,0.0,0.5,-0.3559
Monthly bill: $120 vs $105,-0.2894,0.0092,0.0,0.5,-0.5789
Monthly bill: $125 vs $105,-0.3832,0.0100,0.0,0.5,-0.7664
Percentage of renewables: 30% vs 10%,0.0973,0.0073,0.0,0.5,0.1946
Percentage of renewables: 50% vs 10%,0.1706,0.0081,0.0,0.5,0.3413
Percentage of renewables: 100% vs 10%,0.2421,0.0096,0.0,0.5,0.4842
Locally produced renewables: Yes vs No,0.1042,0.0056,0.0,0.5,0.2085


In [5]:
WU2025_SIGMA_Y

0.5

# Dataset 2: Berke 2023 (drone-delivery conjoint — future train)

In [33]:
# Load Berke 2023 survey rows and filter to paired delivery mode choices (`delivery_choice_v0..v3`)
t_berke0 = time.perf_counter()
berke2023_df = load_berke2023_df()
berke2023_choice_df = berke2023_df[berke2023_df["v"].str.startswith("delivery_choice_v")].reset_index(drop=True)
berke2023_load_sec = time.perf_counter() - t_berke0
if len(berke2023_choice_df) == 0:
    raise ValueError("No `delivery_choice_v*` rows found in berke2023_df")

display(
    pd.Series(
        {
            "total_exported_rows": len(berke2023_df),
            "exported_questions": berke2023_df["v"].nunique(),
            "delivery_choice_tasks": len(berke2023_choice_df),
            "respondents": berke2023_choice_df["CASEID"].nunique(),
            "v0_takeout_food_tasks": int((berke2023_choice_df["v"] == "delivery_choice_v0").sum()),
            "v1_medications_tasks": int((berke2023_choice_df["v"] == "delivery_choice_v1").sum()),
            "v2_liquor_store_tasks": int((berke2023_choice_df["v"] == "delivery_choice_v2").sum()),
            "v3_groceries_tasks": int((berke2023_choice_df["v"] == "delivery_choice_v3").sum()),
            "options_per_task": 2,
            "load_sec": round(berke2023_load_sec, 4),
        },
        name="berke2023_overview",
    ).to_frame()
)

,berke2023_overview
total_exported_rows,54872.0000
exported_questions,9.0000
delivery_choice_tasks,36297.0000
respondents,3625.0000
v0_takeout_food_tasks,9528.0000
v1_medications_tasks,11481.0000
v2_liquor_store_tasks,7269.0000
v3_groceries_tasks,8019.0000
options_per_task,2.0000
load_sec,0.3854


In [34]:
# Berke et al. (2023, Section 3.4 & Table 4) estimate 4 separate vendor-specific models (`v0..v3`),
# and excluded dominated alternatives when building choice pairs (p. 7), so ground and drone attributes are correlated.
# We therefore fit 4 separate respondent-clustered OLS regressions in the stacked option-row format used for Wu 2025:
# - each task gives 2 rows in `one_hot` order (Ground, Drone), y = 1 if that option was chosen
# - each row's attributes are coded as THIS option minus the OTHER option (privacy: ground has none, so it counts as 0)
# - `is_drone` is the drone's alternative-specific constant
# Because the ground row mirrors the drone row (y_ground = 1 - y_drone, all differences flipped), the slopes equal
# the pair-difference regression (`y_drone` on `X_drone - X_ground`) exactly, for human choices and model probabilities alike.
from data.berke2023.stimulus import (
    COST_LEVELS as BERKE_COST_LEVELS,
    TIME_LEVELS as BERKE_TIME_LEVELS,
    VENDOR_TYPES as BERKE_VENDOR_TYPES,
)

BERKE_EXPECTED_OPTIONS = [text for _, text in BERKE_CHOICE_OPTIONS]


def parse_berke_design(qid: str) -> dict[str, int]:
    # `qid` ends in `gc<k>_gt<k>_dc<k>_dt<k>_dp<k>`: ground cost, ground time, drone cost, drone time, drone privacy
    design = {}
    for token in qid.split(":")[-1].split("_"):
        design[token[:2]] = int(token[2:])
    return design


def berke_attribute_codes(cost_idx: int, time_idx: int, privacy: int) -> dict[str, float]:
    # Level dummies vs level 0 ($0 fee, shortest wait), plus the privacy flag (YES = 1)
    return {
        "fee_L1": float(cost_idx == 1),
        "fee_L2": float(cost_idx == 2),
        "fee_L3": float(cost_idx == 3),
        "wait_L1": float(time_idx == 1),
        "wait_L2": float(time_idx == 2),
        "wait_L3": float(time_idx == 3),
        "privacy": float(privacy),
    }


berke_stacked_rows: list[dict] = []
for t_idx, row in enumerate(berke2023_choice_df.itertuples(index=False)):
    if list(row.option_texts) != BERKE_EXPECTED_OPTIONS:
        raise ValueError(f"Berke 2023 task {t_idx}: expected {BERKE_EXPECTED_OPTIONS}, got {row.option_texts}")
    if len(row.one_hot) != 2 or float(sum(row.one_hot)) != 1.0:
        raise ValueError(f"Berke 2023 task {t_idx}: expected a 2-option one-hot choice, got {row.one_hot}")

    # 1. Read the vendor and the two options' design levels from the qid
    v_idx = int(str(row.v).rsplit("v", 1)[-1])
    design = parse_berke_design(str(row.qid))
    ground_codes = berke_attribute_codes(design["gc"], design["gt"], privacy=0)
    drone_codes = berke_attribute_codes(design["dc"], design["dt"], privacy=design["dp"])

    # 2. One row per option, in `one_hot` order (Ground, Drone): own attributes minus the other option's
    options_in_order = [
        {"own_codes": ground_codes, "other_codes": drone_codes, "cost_idx": design["gc"], "time_idx": design["gt"]},
        {"own_codes": drone_codes, "other_codes": ground_codes, "cost_idx": design["dc"], "time_idx": design["dt"]},
    ]
    for opt_idx, option in enumerate(options_in_order):
        own_codes = option["own_codes"]
        other_codes = option["other_codes"]
        stacked_row = {
            "CASEID": row.CASEID,
            "task_idx": t_idx,
            "v_idx": v_idx,
            "vendor": BERKE_VENDOR_TYPES[v_idx][0],
            "option": BERKE_EXPECTED_OPTIONS[opt_idx],
            "fee": BERKE_COST_LEVELS[v_idx][option["cost_idx"]],
            "wait": BERKE_TIME_LEVELS[v_idx][option["time_idx"]],
            "is_drone": float(opt_idx),
        }
        for attr_name in own_codes:
            stacked_row[f"{attr_name}_diff"] = own_codes[attr_name] - other_codes[attr_name]
        stacked_row["y"] = float(row.one_hot[opt_idx])
        berke_stacked_rows.append(stacked_row)

berke2023_opt_df = pd.DataFrame(berke_stacked_rows)

BERKE_STACKED_FORMULA = (
    "y ~ is_drone + privacy_diff + fee_L1_diff + fee_L2_diff + fee_L3_diff "
    "+ wait_L1_diff + wait_L2_diff + wait_L3_diff"
)
BERKE_SLOPE_ROWS = [
    ("privacy_diff", "Drone privacy: YES vs NO"),
    ("fee_L1_diff", "Fee L1 vs $0 ($1)"),
    ("fee_L2_diff", "Fee L2 vs $0 ($3 fast / $5 meds)"),
    ("fee_L3_diff", "Fee L3 vs $0 ($5 fast / $10 meds)"),
    ("wait_L1_diff", "Wait L1 vs L0 (20m fast / 2h meds)"),
    ("wait_L2_diff", "Wait L2 vs L0 (30m fast / 6h meds)"),
    ("wait_L3_diff", "Wait L3 vs L0 (45m fast / 1d meds)"),
]
BERKE_CONSTANT_LABEL = "Drone constant (is_drone; = 2 x P(drone | equal fee/wait, NO priv) - 1)"

# 3. Outcome SD on the stacked option rows (exactly 0.5: every task has one chosen and one unchosen row)
BERKE2023_SIGMA_Y = float(np.std(berke2023_opt_df["y"].to_numpy(), ddof=0))

# 4. Fit one respondent-clustered OLS per vendor
BERKE2023_VENDOR_SIGMA_Y: dict[str, float] = {}
berke_ols_rows: list[dict] = []
berke_vendor_cols: dict[tuple[str, str], dict[str, float]] = {}

for v_idx, (v_label, _) in enumerate(BERKE_VENDOR_TYPES):
    sub = berke2023_opt_df[berke2023_opt_df["v_idx"] == v_idx]
    v_col = f"v{v_idx}: {v_label}"
    sigma_v = float(np.std(sub["y"].to_numpy(), ddof=0))
    BERKE2023_VENDOR_SIGMA_Y[v_col] = sigma_v
    fit = smf.ols(BERKE_STACKED_FORMULA, data=sub).fit(cov_type="cluster", cov_kwds={"groups": sub["CASEID"]})

    beta_col = {BERKE_CONSTANT_LABEL: float(fit.params["is_drone"])}
    se_col = {BERKE_CONSTANT_LABEL: float(fit.bse["is_drone"])}
    for col_key, row_label in BERKE_SLOPE_ROWS:
        b = float(fit.params[col_key])
        se = float(fit.bse[col_key])
        pv = float(fit.pvalues[col_key])
        beta_col[row_label] = b
        se_col[row_label] = se
        berke_ols_rows.append(
            {
                "vendor": v_col,
                "term": f"{v_col} | {row_label}",
                "human_beta": b,
                "cluster_se": se,
                "p_value": pv,
                "sigma_y": sigma_v,
                "std_beta": b / sigma_v,
            }
        )
    beta_col["sigma_y (SD of stacked y)"] = sigma_v
    se_col["sigma_y (SD of stacked y)"] = float("nan")
    berke_vendor_cols[(v_col, "beta")] = beta_col
    berke_vendor_cols[(v_col, "cluster_se")] = se_col

berke2023_human_ols_df = pd.DataFrame(berke_ols_rows).set_index("term")
berke2023_vendor_table_df = pd.DataFrame(berke_vendor_cols)
display(berke2023_vendor_table_df.round(4))

v0: take-out food            v1: prescription medications            v2: liquor store             \
                                                                beta cluster_se                         beta cluster_se             beta cluster_se   
Drone constant (is_drone; = 2 x P(drone | equal...           -0.3967     0.0144                      -0.5213     0.0133          -0.4636     0.0159   
Drone privacy: YES vs NO                                      0.1607     0.0095                       0.2242     0.0088           0.2107     0.0111   
Fee L1 vs $0 ($1)                                            -0.1509     0.0088                      -0.0822     0.0087          -0.1534     0.0114   
Fee L2 vs $0 ($3 fast / $5 meds)                             -0.3298     0.0102                      -0.2964     0.0096          -0.3251     0.0128   
Fee L3 vs $0 ($5 fast / $10 meds)                            -0.4978     0.0108                      -0.5023     0.0108          -0.5268     0.0126   
Wait L1 vs L0 (20m fast / 2h meds)                           -0.0631     0.0107                      -0.0743     0.0095          -0.0242     0.0109   
Wait L2 vs L0 (30m fast / 6h meds)                           -0.2082     0.0106                      -0.1924     0.0095          -0.1521     0.0111   
Wait L3 vs L0 (45m fast / 1d meds)                           -0.3419     0.0124                      -0.2638     0.0102          -0.2772     0.0130   
sigma_y (SD of stacked y)                                     0.5000        NaN                       0.5000        NaN           0.5000        NaN   

                                                   v3: last-minute groceries             
                                                                        beta cluster_se  
Drone constant (is_drone; = 2 x P(drone | equal...                   -0.4711     0.0147  
Drone privacy: YES vs NO                                              0.2138     0.0102  
Fee L1 vs $0 ($1)                                                    -0.1659     0.0117  
Fee L2 vs $0 ($3 fast / $5 meds)                                     -0.3608     0.0123  
Fee L3 vs $0 ($5 fast / $10 meds)                                    -0.5790     0.0145  
Wait L1 vs L0 (20m fast / 2h meds)                                   -0.0709     0.0099  
Wait L2 vs L0 (30m fast / 6h meds)                                   -0.2184     0.0115  
Wait L3 vs L0 (45m fast / 1d meds)                                   -0.3460     0.0135  
sigma_y (SD of stacked y)                                             0.5000        NaN

In [35]:
berke2023_opt_df

,CASEID,task_idx,v_idx,vendor,option,fee,wait,is_drone,fee_L1_diff,fee_L2_diff,fee_L3_diff,wait_L1_diff,wait_L2_diff,wait_L3_diff,privacy_diff,y
0,berke2023_R_2DTxG8oxoUwagCE,0,0,take-out food,Ground vehicle,$0,15 minutes,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,1.0
1,berke2023_R_2DTxG8oxoUwagCE,0,0,take-out food,Drone,$0,30 minutes,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,berke2023_R_11jaEnzddsEMEAY,1,0,take-out food,Ground vehicle,$0,15 minutes,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,1.0
3,berke2023_R_11jaEnzddsEMEAY,1,0,take-out food,Drone,$0,30 minutes,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,berke2023_R_z28iJpRZr3tpK7L,2,0,take-out food,Ground vehicle,$0,15 minutes,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72589,berke2023_R_djwTsWVLmhZHeAV,36294,3,last-minute groceries,Drone,$5,45 minutes,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72590,berke2023_R_BJUbZZyAeNtAn8B,36295,3,last-minute groceries,Ground vehicle,$5,45 minutes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
72591,berke2023_R_BJUbZZyAeNtAn8B,36295,3,last-minute groceries,Drone,$5,45 minutes,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72592,berke2023_R_30x9GGNLfYGhLrG,36296,3,last-minute groceries,Ground vehicle,$5,45 minutes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [36]:
BERKE2023_SIGMA_Y

0.5

In [37]:
BERKE2023_VENDOR_SIGMA_Y

{'v0: take-out food': 0.5,
 'v1: prescription medications': 0.5,
 'v2: liquor store': 0.5,
 'v3: last-minute groceries': 0.5}

# Dataset 3: Kriner 2026 (airline SAF conjoint — future validation)

In [38]:
# Load Kriner 2026 SAF survey rows and filter to paired flight choices (`flight_choice`)
t_kr0 = time.perf_counter()
kriner2026_df = load_kriner2026_df()
kriner2026_choice_df = kriner2026_df[kriner2026_df["v"] == "flight_choice"].reset_index(drop=True)
kriner2026_load_sec = time.perf_counter() - t_kr0
if len(kriner2026_choice_df) == 0:
    raise ValueError("No `flight_choice` rows found in kriner2026_df")

display(
    pd.Series(
        {
            "total_exported_rows": len(kriner2026_df),
            "exported_questions": kriner2026_df["v"].nunique(),
            "flight_choice_tasks": len(kriner2026_choice_df),
            "flight_choice_option_rows": len(kriner2026_choice_df) * 2,
            "respondents": kriner2026_choice_df["CASEID"].nunique(),
            "options_per_task": 2,
            "load_sec": round(kriner2026_load_sec, 4),
        },
        name="kriner2026_overview",
    ).to_frame()
)

,kriner2026_overview
total_exported_rows,9000.0000
exported_questions,5.0000
flight_choice_tasks,5000.0000
flight_choice_option_rows,10000.0000
respondents,1000.0000
options_per_task,2.0000
load_sec,0.0251


In [39]:
# Fit Goldfarb, Kriner et al. (2026) Table 2 respondent-clustered OLS across all 10,000 flight-option rows
KRINER_EXPECTED_OPTIONS = [text for _, text in KRINER_CHOICE_OPTIONS]

kr_opt_rows: list[dict] = []
for t_idx, row in enumerate(kriner2026_choice_df.itertuples(index=False)):
    if list(row.option_texts) != KRINER_EXPECTED_OPTIONS:
        raise ValueError(f"Kriner 2026 task {t_idx}: expected {KRINER_EXPECTED_OPTIONS}, got {row.option_texts}")
    if row.option_descriptions is None or len(row.option_descriptions) != 2:
        raise ValueError(f"Kriner 2026 task {t_idx}: expected 2 option_descriptions")
    for desc, y_val in zip(row.option_descriptions, row.one_hot, strict=True):
        kr_opt_rows.append({"CASEID": row.CASEID, **parse_flight_description(desc), "y": float(y_val)})

kriner2026_opt_df = pd.DataFrame(kr_opt_rows)
KRINER2026_SIGMA_Y = float(np.std(kriner2026_opt_df["y"].to_numpy(), ddof=0))
kr_fit = smf.ols(KRINER_AMCE_FORMULA, data=kriner2026_opt_df).fit(
    cov_type="cluster", cov_kwds={"groups": kriner2026_opt_df["CASEID"]}
)

kriner2026_human_ols_df = pd.DataFrame(
    [
        {
            "term": label,
            "human_beta": float(kr_fit.params[p_term]),
            "cluster_se": float(kr_fit.bse[p_term]),
            "p_value": float(kr_fit.pvalues[p_term]),
            "sigma_y": KRINER2026_SIGMA_Y,
            "std_beta": float(kr_fit.params[p_term]) / KRINER2026_SIGMA_Y,
        }
        for p_term, label in KRINER_AMCE_TERMS
    ]
).set_index("term")

display(kriner2026_human_ols_df.round(4))

,human_beta,cluster_se,p_value,sigma_y,std_beta
term,,,,,
Airline: Frontier,-0.0854,0.0137,0.0000,0.5,-0.1709
Airline: Southwest,-0.0238,0.0139,0.0863,0.5,-0.0476
Airline: United,-0.0220,0.0132,0.0952,0.5,-0.0440
Fuel: Sustainable aviation fuel,0.0691,0.0105,0.0000,0.5,0.1381
Baggage: Checked bag+carry-on,0.0561,0.0124,0.0000,0.5,0.1121
Baggage: Personal item only,-0.1393,0.0122,0.0000,0.5,-0.2785
Space: Extra legroom,0.0653,0.0097,0.0000,0.5,0.1305
Seat: Middle,-0.0906,0.0127,0.0000,0.5,-0.1813
Seat: Window,0.0384,0.0124,0.0019,0.5,0.0769


In [40]:
kriner2026_opt_df

,CASEID,baggage_n,price_n,airline_n,seat_n,legroom_n,fuel_type_n,y
0,kriner2026_1,1,1,4,2,2,2,1.0
1,kriner2026_1,1,1,1,2,1,2,0.0
2,kriner2026_1,3,3,4,1,1,1,0.0
3,kriner2026_1,2,4,1,3,1,2,1.0
4,kriner2026_1,2,3,1,2,2,1,0.0
...,...,...,...,...,...,...,...,...
9995,kriner2026_1000,3,3,2,2,1,1,1.0
9996,kriner2026_1000,3,1,2,1,1,1,1.0
9997,kriner2026_1000,1,4,1,3,2,1,0.0
9998,kriner2026_1000,3,2,2,1,1,2,1.0


In [41]:
KRINER2026_SIGMA_Y

0.5

# Dataset 4: COVID / Kreps et al. 2020 (vaccine conjoint — locked test)

maybe later explore the position effect and how we can replicate this too in the regressions

In [42]:
# Load Kreps et al. (2020) COVID survey rows and filter to 3-option paired vaccine choices (`vax_choice`)
t_cv0 = time.perf_counter()
covid_df = load_covid_df()
covid_choice_df = covid_df[covid_df["v"] == "vax_choice"].reset_index(drop=True)
covid_load_sec = time.perf_counter() - t_cv0
if len(covid_choice_df) == 0:
    raise ValueError("No `vax_choice` rows found in covid_df")

display(
    pd.Series(
        {
            "total_exported_rows": len(covid_df),
            "exported_questions": covid_df["v"].nunique(),
            "vax_choice_tasks": len(covid_choice_df),
            "vax_choice_profile_rows": len(covid_choice_df) * 2,
            "respondents": covid_choice_df["CASEID"].nunique(),
            "options_per_task (incl. neither)": len(covid_choice_df["option_texts"].iloc[0]),
            "load_sec": round(covid_load_sec, 4),
        },
        name="covid_overview",
    ).to_frame()
)

,covid_overview
total_exported_rows,50273.0000
exported_questions,13.0000
vax_choice_tasks,9855.0000
vax_choice_profile_rows,19710.0000
respondents,1971.0000
options_per_task (incl. neither),3.0000
load_sec,0.1122


In [43]:
# Fit Kreps et al. (2020) Table 3 respondent-clustered AMCE OLS on the 2 displayed vaccine profiles per task
cv_opt_rows: list[dict] = []
for t_idx, row in enumerate(covid_choice_df.itertuples(index=False)):
    if len(row.one_hot) != 3:
        raise ValueError(f"COVID task {t_idx}: expected 3 choice options (Vaccine A, B, neither), got {len(row.one_hot)}")
    pa, pb = str(row.qid).split(":")[-1].split("_")
    cv_opt_rows.append({"CASEID": row.CASEID, **covid_profile_attr_dict(pa), "y": float(row.one_hot[0])})
    cv_opt_rows.append({"CASEID": row.CASEID, **covid_profile_attr_dict(pb), "y": float(row.one_hot[1])})

covid_opt_df = pd.DataFrame(cv_opt_rows)
COVID_SIGMA_Y = float(np.std(covid_opt_df["y"].to_numpy(), ddof=0))
cv_fit = smf.ols(COVID_AMCE_FORMULA, data=covid_opt_df).fit(
    cov_type="cluster", cov_kwds={"groups": covid_opt_df["CASEID"]}
)

covid_human_ols_df = pd.DataFrame(
    [
        {
            "term": label,
            "human_beta": float(cv_fit.params[p_term]),
            "cluster_se": float(cv_fit.bse[p_term]),
            "p_value": float(cv_fit.pvalues[p_term]),
            "sigma_y": COVID_SIGMA_Y,
            "std_beta": float(cv_fit.params[p_term]) / COVID_SIGMA_Y,
        }
        for p_term, label in COVID_AMCE_TERMS
    ]
).set_index("term")

display(covid_human_ols_df.round(4))

,human_beta,cluster_se,p_value,sigma_y,std_beta
term,,,,,
Efficacy: 70% vs 50%,0.0744,0.0086,0.0000,0.4888,0.1521
Efficacy: 90% vs 50%,0.1641,0.0094,0.0000,0.4888,0.3357
Duration: 5y vs 1y,0.0541,0.0071,0.0000,0.4888,0.1106
Severe side effects: 1/1M vs 1/10k,0.0653,0.0071,0.0000,0.4888,0.1337
Mild side effects: 1/30 vs 1/10,0.0125,0.0069,0.0714,0.4888,0.0256
FDA: EUA vs Full approval,-0.0287,0.0070,0.0000,0.4888,-0.0588
Origin: UK vs US,-0.0380,0.0087,0.0000,0.4888,-0.0777
Origin: China vs US,-0.1308,0.0090,0.0000,0.4888,-0.2675
Endorsed: Biden vs Trump,0.0181,0.0108,0.0935,0.4888,0.0371


In [44]:
COVID_SIGMA_Y

0.4888348858022344

In [45]:
covid_opt_df

,CASEID,efficacy_n,duration_n,majorside_n,minorside_n,fda_n,origin_n,endorsed_n,y
0,covid_1,3,2,1,2,1,2,1,1.0
1,covid_1,2,1,2,1,1,3,4,0.0
2,covid_1,3,2,1,1,1,2,3,1.0
3,covid_1,1,2,1,1,2,1,4,0.0
4,covid_1,1,2,2,2,2,1,2,0.0
...,...,...,...,...,...,...,...,...,...
19705,covid_1971,1,1,2,1,1,2,4,0.0
19706,covid_1971,3,1,2,2,2,3,4,0.0
19707,covid_1971,3,2,2,1,2,2,4,0.0
19708,covid_1971,1,1,1,2,2,1,3,0.0


# Dataset 5 (v5 Active Train): Grummon 2023 (FOPL vending-machine RCT)

In [52]:
# Load Grummon 2023 RCT rows, build the 16-beverage / 16-snack calorie catalog, and keep only primary choice tasks
t_g23_0 = time.perf_counter()
grummon2023_df = load_grummon2023_df()
_, _g23_bev_chars, _g23_snack_chars = load_grummon2023_analytic_cohort()
grummon2023_catalog = build_grummon2023_product_catalog(_g23_bev_chars, _g23_snack_chars)

# Keep respondents who answered both primary vending-machine choice tasks (`bev_choice` and `snack_choice`), dropping all 30 Likert items
g23_bev_all = grummon2023_df[grummon2023_df["v"] == "bev_choice"].set_index("CASEID")
g23_snack_all = grummon2023_df[grummon2023_df["v"] == "snack_choice"].set_index("CASEID")
# Sort CASEIDs numerically (`grummon2023_<id>`) so respondents are not grouped by experimental arm
g23_complete_caseids = sorted(
    g23_bev_all.index.intersection(g23_snack_all.index),
    key=lambda cid: int(str(cid).rsplit("_", 1)[-1]),
)
g23_bev_df = g23_bev_all.loc[g23_complete_caseids].reset_index()
g23_snack_df = g23_snack_all.loc[g23_complete_caseids].reset_index()
if not (g23_bev_df["arm"].to_numpy() == g23_snack_df["arm"].to_numpy()).all():
    raise AssertionError("Mismatch in respondent FOPL arm assignment between bev_choice and snack_choice")
grummon2023_load_sec = time.perf_counter() - t_g23_0

display(
    pd.Series(
        {
            "total_exported_rows": len(grummon2023_df),
            "exported_questions (2 choice + 30 Likert)": grummon2023_df["v"].nunique(),
            "complete_choice_respondents": len(g23_complete_caseids),
            "primary_choice_tasks (bev + snack)": len(g23_complete_caseids) * 2,
            "options_per_vending_machine": 16,
            "randomized_fopl_arms": g23_bev_df["arm"].nunique(),
            "load_sec": round(grummon2023_load_sec, 4),
        },
        name="grummon2023_overview",
    ).to_frame()
)

,grummon2023_overview
total_exported_rows,254145.0000
exported_questions (2 choice + 30 Likert),32.0000
complete_choice_respondents,7943.0000
primary_choice_tasks (bev + snack),15886.0000
options_per_vending_machine,16.0000
randomized_fopl_arms,5.0000
load_sec,0.6894


In [53]:
# Extract ordered 16-product calorie vectors and precompute fixed human training SDs (`sigma_bev`, `sigma_snack`)
GRUMMON2023_TREAT_ARMS = ["Green", "TrafficLight", "PhysicalActivity", "Warning"]
GRUMMON2023_ARM_ORDER = ["Calorie", *GRUMMON2023_TREAT_ARMS]
if list(GRUMMON2023_ARM_NAMES.values()) != GRUMMON2023_ARM_ORDER:
    raise AssertionError(f"Unexpected Grummon 2023 arm order: {list(GRUMMON2023_ARM_NAMES.values())}")

BEV_KCAL_VEC = np.array([float(item["true_cal"]) for item in grummon2023_catalog["bev"]], dtype=np.float64)
SNACK_KCAL_VEC = np.array([float(item["true_cal"]) for item in grummon2023_catalog["snack"]], dtype=np.float64)
if BEV_KCAL_VEC.shape != (16,) or SNACK_KCAL_VEC.shape != (16,):
    raise ValueError(f"Expected 16-item calorie vectors, got {BEV_KCAL_VEC.shape} and {SNACK_KCAL_VEC.shape}")

# Map each respondent's chosen product code (1..16) to calories selected
g23_bev_ans_idx = g23_bev_df["answer_code"].to_numpy(dtype=int) - 1
g23_snack_ans_idx = g23_snack_df["answer_code"].to_numpy(dtype=int) - 1
if ((g23_bev_ans_idx < 0) | (g23_bev_ans_idx >= 16) | (g23_snack_ans_idx < 0) | (g23_snack_ans_idx >= 16)).any():
    raise ValueError("Grummon 2023 answer_code outside expected 1..16 product codes")

g23_bev_df["kcal"] = BEV_KCAL_VEC[g23_bev_ans_idx]
g23_snack_df["kcal"] = SNACK_KCAL_VEC[g23_snack_ans_idx]

# Fixed training-set outcome standard deviations (Caveat 2: computed once on full human training data)
GRUMMON2023_SIGMA_BEV = float(np.std(g23_bev_df["kcal"].to_numpy(), ddof=0))
GRUMMON2023_SIGMA_SNACK = float(np.std(g23_snack_df["kcal"].to_numpy(), ddof=0))
if GRUMMON2023_SIGMA_BEV <= 0.0 or GRUMMON2023_SIGMA_SNACK <= 0.0:
    raise ValueError("Non-positive Grummon 2023 calorie standard deviation")

# Fit Grummon et al. (2023): TWO separate HC1 OLS models, one per primary outcome (paper: "calories selected from
# beverages and from snacks"), each `kcal ~ C(arm, Treatment(reference='Calorie'))` on its own task's choices
G23_ARM_FORMULA = "kcal ~ C(arm, Treatment(reference='Calorie'))"
g23_bev_fit = smf.ols(G23_ARM_FORMULA, data=g23_bev_df).fit(cov_type="HC1")
g23_snack_fit = smf.ols(G23_ARM_FORMULA, data=g23_snack_df).fit(cov_type="HC1")

GRUMMON2023_BETA_NAMES = [
    *(f"Beverage: {arm} vs Calorie" for arm in GRUMMON2023_TREAT_ARMS),
    *(f"Snack: {arm} vs Calorie" for arm in GRUMMON2023_TREAT_ARMS),
]
G23_MODELS = [
    ("Model 1: beverage calories (bev_choice)", "Beverage", g23_bev_fit, GRUMMON2023_SIGMA_BEV, len(g23_bev_df)),
    ("Model 2: snack calories (snack_choice)", "Snack", g23_snack_fit, GRUMMON2023_SIGMA_SNACK, len(g23_snack_df)),
]
G23_INTERCEPT_LABEL = "Intercept (Calorie arm mean kcal)"

# Long table (one row per beta, Beverage then Snack): the order the training loss concatenates the 8 betas in
g23_ols_rows: list[dict] = []
# Wide display table: one column group per model, so the two separate regressions are visible side by side
g23_model_cols: dict[tuple[str, str], dict[str, float]] = {}

for model_label, task_label, fit_obj, sigma_kcal, n_resp in G23_MODELS:
    beta_col = {G23_INTERCEPT_LABEL: float(fit_obj.params["Intercept"])}
    se_col = {G23_INTERCEPT_LABEL: float(fit_obj.bse["Intercept"])}
    std_col = {G23_INTERCEPT_LABEL: float("nan")}
    for arm in GRUMMON2023_TREAT_ARMS:
        p_term = f"C(arm, Treatment(reference='Calorie'))[T.{arm}]"
        kcal_beta = float(fit_obj.params[p_term])
        kcal_se = float(fit_obj.bse[p_term])
        std_beta = kcal_beta / sigma_kcal
        g23_ols_rows.append(
            {
                "term": f"{task_label}: {arm} vs Calorie",
                "model": model_label,
                "human_beta_kcal": kcal_beta,
                "hc1_se_kcal": kcal_se,
                "p_value": float(fit_obj.pvalues[p_term]),
                "sigma_y_kcal": sigma_kcal,
                "std_beta": std_beta,
            }
        )
        arm_label = f"{arm} vs Calorie"
        beta_col[arm_label] = kcal_beta
        se_col[arm_label] = kcal_se
        std_col[arm_label] = std_beta
    beta_col["sigma_y (SD of kcal)"] = sigma_kcal
    se_col["sigma_y (SD of kcal)"] = float("nan")
    std_col["sigma_y (SD of kcal)"] = float("nan")
    beta_col["N respondents"] = float(n_resp)
    se_col["N respondents"] = float("nan")
    std_col["N respondents"] = float("nan")
    g23_model_cols[(model_label, "beta_kcal")] = beta_col
    g23_model_cols[(model_label, "hc1_se_kcal")] = se_col
    g23_model_cols[(model_label, "std_beta")] = std_col

grummon2023_human_ols_df = pd.DataFrame(g23_ols_rows).set_index("term")
grummon2023_model_table_df = pd.DataFrame(g23_model_cols)
display(grummon2023_model_table_df.round(4))

Model 1: beverage calories (bev_choice)                      Model 2: snack calories (snack_choice)                     
                                                                beta_kcal hc1_se_kcal std_beta                              beta_kcal hc1_se_kcal std_beta
Intercept (Calorie arm mean kcal)                                125.8103      2.9284      NaN                               182.7921      1.7348      NaN
Green vs Calorie                                                 -34.1531      4.0874  -0.2968                               -12.7322      2.3247  -0.1923
TrafficLight vs Calorie                                          -31.4715      4.1171  -0.2735                               -13.6485      2.3449  -0.2061
PhysicalActivity vs Calorie                                      -39.0136      4.0531  -0.3390                               -18.4529      2.4394  -0.2787
Warning vs Calorie                                               -28.1985      4.0934  -0.2450                               -14.2004      2.4201  -0.2145
sigma_y (SD of kcal)                                             115.0825         NaN      NaN                                66.2099         NaN      NaN
N respondents                                                   7943.0000         NaN      NaN                              7943.0000         NaN      NaN

In [54]:
GRUMMON2023_SIGMA_BEV

115.08247547828087

In [55]:
GRUMMON2023_SIGMA_SNACK

66.20991831500943

# Dataset 6 (v5 Active Test): Grummon 2025 (FOPL paired-choice RCT)

In [56]:
# Load Grummon 2025 RCT rows and filter to the 6 primary healthier-product identification choices (`idbev..idmeal`)
t_g25_0 = time.perf_counter()
grummon2025_df = load_grummon2025_df()
grummon2025_choice_df = grummon2025_df[grummon2025_df["v"].isin(GRUMMON2025_PRIMARY_STEMS)].reset_index(drop=True)
grummon2025_load_sec = time.perf_counter() - t_g25_0
if len(grummon2025_choice_df) == 0:
    raise ValueError("No primary choice rows found in grummon2025_df")

g25_items_per_resp = grummon2025_choice_df.groupby("CASEID").size()
if int(g25_items_per_resp.min()) != 6 or int(g25_items_per_resp.max()) != 6:
    raise ValueError("Expected every Grummon 2025 respondent to have all 6 primary `id*` choice items")

display(
    pd.Series(
        {
            "total_exported_rows": len(grummon2025_df),
            "exported_questions": grummon2025_df["v"].nunique(),
            "primary_choice_tasks": len(grummon2025_choice_df),
            "respondents": grummon2025_choice_df["CASEID"].nunique(),
            "primary_tasks_per_respondent": 6,
            "options_per_task": 3,
            "randomized_fopl_arms": grummon2025_choice_df["arm"].nunique(),
            "load_sec": round(grummon2025_load_sec, 4),
        },
        name="grummon2025_overview",
    ).to_frame()
)

,grummon2025_overview
total_exported_rows,114899.0000
exported_questions,21.0000
primary_choice_tasks,33798.0000
respondents,5633.0000
primary_tasks_per_respondent,6.0000
options_per_task,3.0000
randomized_fopl_arms,6.0000
load_sec,0.5694


In [57]:
# Fit Grummon et al. (2025) participant-level HC1 OLS on average healthier-product selection (`option 0`) vs `Positive`
g25_work = grummon2025_choice_df[["CASEID", "arm"]].copy()
g25_work["chose_healthier"] = [float(oh[0]) for oh in grummon2025_choice_df["one_hot"]]
g25_part_df = g25_work.groupby("CASEID", as_index=False).agg(
    arm=("arm", "first"),
    human=("chose_healthier", "mean"),
)

GRUMMON2025_SIGMA_Y = float(np.std(g25_part_df["human"].to_numpy(), ddof=0))
g25_fit = smf.ols("human ~ C(arm, Treatment(reference='Positive'))", data=g25_part_df).fit(cov_type="HC1")

GRUMMON2025_BETA_NAMES = list(GRUMMON2025_TREAT_ARMS)
grummon2025_human_ols_df = pd.DataFrame(
    [
        {
            "arm_vs_Positive": arm,
            "human_beta": float(g25_fit.params[f"C(arm, Treatment(reference='Positive'))[T.{arm}]"]),
            "hc1_se": float(g25_fit.bse[f"C(arm, Treatment(reference='Positive'))[T.{arm}]"]),
            "p_value": float(g25_fit.pvalues[f"C(arm, Treatment(reference='Positive'))[T.{arm}]"]),
            "sigma_y": GRUMMON2025_SIGMA_Y,
            "std_beta": float(g25_fit.params[f"C(arm, Treatment(reference='Positive'))[T.{arm}]"]) / GRUMMON2025_SIGMA_Y,
        }
        for arm in GRUMMON2025_BETA_NAMES
    ]
).set_index("arm_vs_Positive")

display(grummon2025_human_ols_df.round(4))

,human_beta,hc1_se,p_value,sigma_y,std_beta
arm_vs_Positive,,,,,
NutritionInfo,0.1844,0.0119,0.0,0.2704,0.6821
HighIn,0.0541,0.0122,0.0,0.2704,0.2001
Positive+NutritionInfo,0.1788,0.0120,0.0,0.2704,0.6614
Positive+HighIn,0.1194,0.0122,0.0,0.2704,0.4418
Spectrum,0.1083,0.0125,0.0,0.2704,0.4006


In [58]:
GRUMMON2025_SIGMA_Y

0.27037886013736406

In [59]:
g25_part_df['human']

0       0.500000
1       1.000000
2       0.833333
3       1.000000
4       1.000000
          ...   
5628    0.500000
5629    1.000000
5630    0.500000
5631    1.000000
5632    0.500000
Name: human, Length: 5633, dtype: float64

# Cross-dataset beta scale comparison and Grummon 2023 sample-size SNR

In [50]:
# Compare raw vs sigma_y-standardized human beta mean-square across all 6 cleaned experimental choice datasets
datasets_scale_summary_df = pd.DataFrame(
    [
        {
            "dataset": "Wu 2025 (plan_choice)",
            "role_in_v5": "loaded (future train)",
            "respondents": wu2025_choice_df["CASEID"].nunique(),
            "choice_tasks": len(wu2025_choice_df),
            "options_per_task": 2,
            "num_betas": len(wu2025_human_ols_df),
            "outcome_unit": "choice prob [0,1]",
            "mean_sigma_y": round(WU2025_SIGMA_Y, 4),
            "raw_signal_beta_ms": round(float((wu2025_human_ols_df["human_beta"] ** 2).mean()), 5),
            "std_signal_beta_ms": round(float((wu2025_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
        {
            "dataset": "Berke 2023 (4 vendor models)",
            "role_in_v5": "loaded (future train)",
            "respondents": berke2023_choice_df["CASEID"].nunique(),
            "choice_tasks": len(berke2023_choice_df),
            "options_per_task": 2,
            "num_betas": len(berke2023_human_ols_df),
            "outcome_unit": "choice prob [0,1]",
            "mean_sigma_y": round(BERKE2023_SIGMA_Y, 4),
            "raw_signal_beta_ms": round(float((berke2023_human_ols_df["human_beta"] ** 2).mean()), 5),
            "std_signal_beta_ms": round(float((berke2023_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
        {
            "dataset": "Kriner 2026 (flight_choice)",
            "role_in_v5": "loaded (future val)",
            "respondents": kriner2026_choice_df["CASEID"].nunique(),
            "choice_tasks": len(kriner2026_choice_df),
            "options_per_task": 2,
            "num_betas": len(kriner2026_human_ols_df),
            "outcome_unit": "choice prob [0,1]",
            "mean_sigma_y": round(KRINER2026_SIGMA_Y, 4),
            "raw_signal_beta_ms": round(float((kriner2026_human_ols_df["human_beta"] ** 2).mean()), 5),
            "std_signal_beta_ms": round(float((kriner2026_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
        {
            "dataset": "COVID / Kreps 2020 (vax_choice)",
            "role_in_v5": "loaded (locked test)",
            "respondents": covid_choice_df["CASEID"].nunique(),
            "choice_tasks": len(covid_choice_df),
            "options_per_task": 3,
            "num_betas": len(covid_human_ols_df),
            "outcome_unit": "choice prob [0,1]",
            "mean_sigma_y": round(COVID_SIGMA_Y, 4),
            "raw_signal_beta_ms": round(float((covid_human_ols_df["human_beta"] ** 2).mean()), 5),
            "std_signal_beta_ms": round(float((covid_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
        {
            "dataset": "Grummon 2023 (bev + snack)",
            "role_in_v5": "ACTIVE TRAIN",
            "respondents": len(g23_complete_caseids),
            "choice_tasks": len(g23_complete_caseids) * 2,
            "options_per_task": 16,
            "num_betas": len(grummon2023_human_ols_df),
            "outcome_unit": "kcal [0,400]",
            "mean_sigma_y": round(0.5 * (GRUMMON2023_SIGMA_BEV + GRUMMON2023_SIGMA_SNACK), 2),
            "raw_signal_beta_ms": round(float((grummon2023_human_ols_df["human_beta_kcal"] ** 2).mean()), 2),
            "std_signal_beta_ms": round(float((grummon2023_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
        {
            "dataset": "Grummon 2025 (6 primary id*)",
            "role_in_v5": "ACTIVE TEST",
            "respondents": len(g25_part_df),
            "choice_tasks": len(grummon2025_choice_df),
            "options_per_task": 3,
            "num_betas": len(grummon2025_human_ols_df),
            "outcome_unit": "mean healthy prob [0,1]",
            "mean_sigma_y": round(GRUMMON2025_SIGMA_Y, 4),
            "raw_signal_beta_ms": round(float((grummon2025_human_ols_df["human_beta"] ** 2).mean()), 5),
            "std_signal_beta_ms": round(float((grummon2025_human_ols_df["std_beta"] ** 2).mean()), 5),
        },
    ]
).set_index("dataset")

display(datasets_scale_summary_df)

,role_in_v5,respondents,choice_tasks,options_per_task,num_betas,outcome_unit,mean_sigma_y,raw_signal_beta_ms,std_signal_beta_ms
dataset,,,,,,,,,
Wu 2025 (plan_choice),loaded (future train),1861,18591,2,8,"choice prob [0,1]",0.5000,0.04746,0.18986
Berke 2023 (4 vendor models),loaded (future train),3625,36297,2,28,"choice prob [0,1]",0.5000,0.08363,0.33452
Kriner 2026 (flight_choice),loaded (future val),1000,5000,2,12,"choice prob [0,1]",0.5000,0.01562,0.06248
COVID / Kreps 2020 (vax_choice),loaded (locked test),1971,9855,3,11,"choice prob [0,1]",0.4888,0.00642,0.02688
Grummon 2023 (bev + snack),ACTIVE TRAIN,7943,15886,16,8,"kcal [0,400]",90.6500,670.58000,0.06762
Grummon 2025 (6 primary id*),ACTIVE TEST,5633,33798,3,5,"mean healthy prob [0,1]",0.2704,0.01898,0.25967


In [51]:
# Empirical Caveat-3 diagnostic: Monte Carlo sample-beta target noise and SNR on Grummon 2023 across respondent sample sizes
OLS_SVD_RCOND = 1e-8

g23_arms_np = g23_bev_df["arm"].to_numpy()
X_g23_arm_np = np.column_stack(
    [np.ones(len(g23_complete_caseids)), *[(g23_arms_np == arm).astype(np.float64) for arm in GRUMMON2023_TREAT_ARMS]]
)
g23_bev_std_np = g23_bev_df["kcal"].to_numpy(dtype=np.float64) / GRUMMON2023_SIGMA_BEV
g23_snack_std_np = g23_snack_df["kcal"].to_numpy(dtype=np.float64) / GRUMMON2023_SIGMA_SNACK
g23_pop_std_betas_np = grummon2023_human_ols_df["std_beta"].to_numpy(dtype=np.float64)
G23_STD_SIGNAL_MS = float(np.mean(g23_pop_std_betas_np**2))

_snr_rng = np.random.default_rng(42)
_snr_rows: list[dict] = []
for cand_s in (16, 32, 64, 128, 256):
    n_mc = 4000
    rank_def = 0
    noise_vals: list[float] = []
    for _ in range(n_mc):
        idx = _snr_rng.choice(len(g23_complete_caseids), size=cand_s, replace=False)
        Xs = X_g23_arm_np[idx]
        svals = np.linalg.svdvals(Xs)
        if float(svals[-1]) <= OLS_SVD_RCOND * float(svals[0]):
            rank_def += 1
            continue
        A_s = np.linalg.pinv(Xs)[1:]
        b_samp = np.concatenate([A_s @ g23_bev_std_np[idx], A_s @ g23_snack_std_np[idx]])
        noise_vals.append(float(np.mean((b_samp - g23_pop_std_betas_np) ** 2)))
    mean_noise = float(np.mean(noise_vals))
    _snr_rows.append(
        {
            "sample_size": cand_s,
            "steps_per_epoch": math.ceil(len(g23_complete_caseids) / cand_s),
            "mean_resp_per_arm": round(cand_s / 5.0, 1),
            "rank_deficient_pct": round(100.0 * rank_def / n_mc, 2),
            "std_signal_beta_ms": round(G23_STD_SIGNAL_MS, 5),
            "std_target_noise_mse": round(mean_noise, 5),
            "snr_signal_to_noise": round(G23_STD_SIGNAL_MS / mean_noise, 2),
        }
    )

grummon2023_snr_df = pd.DataFrame(_snr_rows).set_index("sample_size")
display(grummon2023_snr_df)

,steps_per_epoch,mean_resp_per_arm,rank_deficient_pct,std_signal_beta_ms,std_target_noise_mse,snr_signal_to_noise
sample_size,,,,,,
16,497,3.2,14.68,0.06762,0.82075,0.08
32,249,6.4,0.45,0.06762,0.37234,0.18
64,125,12.8,0.00,0.06762,0.16878,0.40
128,63,25.6,0.00,0.06762,0.07853,0.86
256,32,51.2,0.00,0.06762,0.03926,1.72


In [16]:
# Differentiable PyTorch OLS operators and Grummon 2023 combined loss (`KL + beta_lambda * standardized_OLS_beta_MSE`)
def encode_grummon2023_arm_row(arm: str) -> list[float]:
    # Convert one respondent's FOPL arm into [1.0 (Calorie intercept), Green, TrafficLight, PhysicalActivity, Warning]
    if arm not in GRUMMON2023_ARM_ORDER:
        raise ValueError(f"Unexpected Grummon 2023 arm {arm!r}; expected one of {GRUMMON2023_ARM_ORDER}")
    return [1.0, *[1.0 if arm == t_arm else 0.0 for t_arm in GRUMMON2023_TREAT_ARMS]]


def build_ols_projector(X: torch.Tensor) -> torch.Tensor:
    # Build fixed linear OLS operator A of shape (K, N) mapping N respondent outcomes -> K non-intercept betas
    if X.ndim != 2 or X.shape[0] < X.shape[1] or X.shape[1] < 2:
        raise ValueError(f"Expected 2D full-column design matrix X with N >= P >= 2, got shape {tuple(X.shape)}")
    X64 = X.to(dtype=torch.float64)
    svals = torch.linalg.svdvals(X64)
    if float(svals[-1]) <= OLS_SVD_RCOND * float(svals[0]):
        raise ValueError(f"Rank-deficient design matrix X: min_sval={float(svals[-1]):.3e}, max_sval={float(svals[0]):.3e}")
    return torch.linalg.pinv(X64)[1:]


def fit_ols_betas(A: torch.Tensor, outcome: torch.Tensor) -> torch.Tensor:
    # Linear OLS projection: beta = A @ outcome (differentiable whenever `outcome` comes from model probabilities)
    if outcome.ndim != 1 or outcome.shape[0] != A.shape[1]:
        raise ValueError(f"Expected 1D outcome of length {A.shape[1]}, got shape {tuple(outcome.shape)}")
    return A @ outcome.to(dtype=A.dtype)


def compute_grummon2023_loss(
    task_logits: torch.Tensor,
    y_onehot: torch.Tensor,
    bev_std_human: torch.Tensor,
    snack_std_human: torch.Tensor,
    A_arm: torch.Tensor,
    bev_std_weights: torch.Tensor,
    snack_std_weights: torch.Tensor,
    beta_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Each respondent contributes 2 consecutive 16-option tasks: even rows = `bev_choice`, odd rows = `snack_choice`
    num_resp = bev_std_human.shape[0]
    if task_logits.shape != (2 * num_resp, 16) or y_onehot.shape != (2 * num_resp, 16):
        raise ValueError(
            f"Expected ({2 * num_resp}, 16) task_logits/y_onehot, got {tuple(task_logits.shape)} and {tuple(y_onehot.shape)}"
        )
    if snack_std_human.shape != (num_resp,) or A_arm.shape != (4, num_resp):
        raise ValueError(f"Shape mismatch: snack_std_human={tuple(snack_std_human.shape)}, A_arm={tuple(A_arm.shape)}")
    if bev_std_weights.shape != (16,) or snack_std_weights.shape != (16,):
        raise ValueError("Expected 16-option standardized calorie weight vectors for beverage and snack")
    if any(t.requires_grad for t in (y_onehot, bev_std_human, snack_std_human, A_arm, bev_std_weights, snack_std_weights)):
        raise ValueError("Human targets, standardized product weights, and OLS projector A_arm must have requires_grad=False")

    # 1. Per-choice KL divergence KL(y || p) averaged across all 2 * S 16-option vending-machine tasks
    log_probs = torch.log_softmax(task_logits, dim=-1)
    kl_loss = torch.nn.functional.kl_div(log_probs, y_onehot, reduction="batchmean")

    # 2. When beta_lambda == 0, compute standardized sample OLS beta MSE under no_grad for logging only
    if beta_lambda == 0.0:
        with torch.no_grad():
            probs_det = torch.softmax(task_logits.detach(), dim=-1)
            pred_betas = torch.cat(
                [
                    fit_ols_betas(A_arm, probs_det[0::2] @ bev_std_weights),
                    fit_ols_betas(A_arm, probs_det[1::2] @ snack_std_weights),
                ],
                dim=0,
            )
            human_betas = torch.cat(
                [fit_ols_betas(A_arm, bev_std_human), fit_ols_betas(A_arm, snack_std_human)],
                dim=0,
            )
            beta_loss = (pred_betas - human_betas).pow(2).mean()
        return kl_loss, kl_loss, beta_loss

    # 3. Differentiable expected standardized calories: sum_j p_{i,j} * (kcal_j / sigma_y) -> 8 sample arm betas
    pred_probs = torch.softmax(task_logits, dim=-1)
    pred_bev_std = pred_probs[0::2] @ bev_std_weights
    pred_snack_std = pred_probs[1::2] @ snack_std_weights
    pred_betas = torch.cat([fit_ols_betas(A_arm, pred_bev_std), fit_ols_betas(A_arm, pred_snack_std)], dim=0)
    human_betas = torch.cat([fit_ols_betas(A_arm, bev_std_human), fit_ols_betas(A_arm, snack_std_human)], dim=0)
    beta_loss = (pred_betas - human_betas).pow(2).mean()

    total_loss = kl_loss + beta_lambda * beta_loss
    return total_loss, kl_loss, beta_loss


# Verify machine-precision parity between PyTorch `build_ols_projector` and `statsmodels` on all 7,943 respondents
_A_g23_full = build_ols_projector(torch.from_numpy(X_g23_arm_np))
_pt_g23_std_betas = torch.cat(
    [
        fit_ols_betas(_A_g23_full, torch.from_numpy(g23_bev_std_np)),
        fit_ols_betas(_A_g23_full, torch.from_numpy(g23_snack_std_np)),
    ],
    dim=0,
)
_max_g23_ols_diff = float((_pt_g23_std_betas.numpy() - g23_pop_std_betas_np). __abs__().max())
if _max_g23_ols_diff > 1e-12:
    raise AssertionError(f"PyTorch Grummon 2023 OLS differs from statsmodels by {_max_g23_ols_diff:.3e}")
print(f"Verified PyTorch Grummon 2023 standardized OLS projector (8 betas) | max diff vs statsmodels = {_max_g23_ols_diff:.2e}")

Verified PyTorch Grummon 2023 standardized OLS projector (8 betas) | max diff vs statsmodels = 4.50e-15


# Model, Grummon 2025 test evaluator, and Grummon 2023 respondent tokenization

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Build `jaredpalmer/kev-0.8b` with bf16 frozen backbone + rank-64 LoRA + pointer head
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    "jaredpalmer/kev-0.8b",
    device=device,
    lora_rank=64,
    weights_dtype="bf16",
)
kev_model.net.head.temperature = 1.0  # training uses raw uncalibrated temperature = 1.0
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": "jaredpalmer/kev-0.8b",
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

In [ ]:
# Build held-out Grummon 2025 test bundle across all 33,798 primary `id*` choice tasks (5,633 respondents, 6 tasks each)
t_g25_bundle0 = time.perf_counter()
grummon2025_records = build_long_records(
    grummon2025_choice_df,
    persona_cols=GRUMMON2025_PERSONA_COLS,
    include_country=False,
    year="grummon2025",
)
grummon2025_eval_bundle = group_records_by_state(
    grummon2025_records,
    view_fn=make_view,
    name="grummon2025_primary_choices",
    valid=True,
)

# Map each record in `grummon2025_eval_bundle["records"]` to its respondent index so 6-item averaging is O(N) tensor scatter
g25_case_arms = grummon2025_choice_df.groupby("CASEID", sort=True)["arm"].first()
g25_case_to_idx = {str(cid): idx for idx, cid in enumerate(g25_case_arms.index)}
g25_rec_case_idx_list: list[int] = []
g25_y_3col_rows: list[list[float]] = []
for rec_idx, rec in enumerate(grummon2025_eval_bundle["records"]):
    h_dist = [float(v) for v in rec["human_dist"]]
    if len(h_dist) != 3 or sorted(h_dist) != [0.0, 0.0, 1.0]:
        raise ValueError(f"Grummon 2025 record {rec_idx}: expected binary 3-option one-hot human_dist, got {h_dist}")
    g25_rec_case_idx_list.append(g25_case_to_idx[str(rec["CASEID"])])
    g25_y_3col_rows.append(h_dist)

g25_rec_case_idx = torch.tensor(g25_rec_case_idx_list, dtype=torch.long)
if not torch.equal(torch.bincount(g25_rec_case_idx, minlength=len(g25_case_arms)), torch.full((len(g25_case_arms),), 6)):
    raise AssertionError("Every Grummon 2025 respondent must appear exactly 6 times in grummon2025_eval_bundle['records']")

y_grummon2025_3col = torch.tensor(g25_y_3col_rows, dtype=torch.float64)
X_grummon2025_pop = torch.tensor(
    np.column_stack(
        [
            np.ones(len(g25_case_arms)),
            *[(g25_case_arms.to_numpy() == arm).astype(np.float64) for arm in GRUMMON2025_BETA_NAMES],
        ]
    ),
    dtype=torch.float64,
)
A_grummon2025_pop = build_ols_projector(X_grummon2025_pop)

# Aggregate human healthier-choice indicators (`option 0`) across the 6 items per respondent and verify exact parity
u_grummon2025_human = torch.zeros(len(g25_case_arms), dtype=torch.float64)
u_grummon2025_human.index_add_(0, g25_rec_case_idx, y_grummon2025_3col[:, 0] / 6.0)
grummon2025_human_betas = fit_ols_betas(A_grummon2025_pop, u_grummon2025_human)
if bool((grummon2025_human_betas == 0).any()):
    raise ValueError("Zero Grummon 2025 human ATE encountered; MAPE denominator would be undefined")

_, _ref_g25_df = evaluate_grummon2025_ols(
    grummon2025_eval_bundle,
    [r["human_dist"] for r in grummon2025_eval_bundle["records"]],
)
_ref_g25_vec = torch.tensor([float(_ref_g25_df.loc[arm, "human"]) for arm in GRUMMON2025_BETA_NAMES], dtype=torch.float64)
_max_g25_diff = float((grummon2025_human_betas - _ref_g25_vec).abs().max())
if _max_g25_diff > 1e-12:
    raise AssertionError(f"PyTorch Grummon 2025 OLS differs from evaluate_grummon2025_ols by {_max_g25_diff:.3e}")
g25_bundle_prep_sec = time.perf_counter() - t_g25_bundle0

print(
    f"Grummon 2025 test bundle: {len(grummon2025_eval_bundle['records']):,} tasks | "
    f"{len(grummon2025_eval_bundle['packed_views']):,} packed persona states | "
    f"A_grummon2025_pop shape: {tuple(A_grummon2025_pop.shape)} | max_diff vs evaluate_grummon2025_ols: {_max_g25_diff:.2e} | "
    f"prep_sec: {g25_bundle_prep_sec:.3f}s"
)
display(pd.Series(grummon2025_human_betas.numpy(), index=GRUMMON2025_BETA_NAMES, name="grummon2025_human_pop_beta").round(4).to_frame())

In [ ]:
@torch.no_grad()
def validate_grummon2025_population_ols(
    model: KevPointerModel,
    eval_bundle: dict,
    rec_case_idx: torch.Tensor,
    A_eval: torch.Tensor,
    y_eval_3col: torch.Tensor,
    human_eval_betas: torch.Tensor,
    beta_names: list[str],
) -> tuple[dict[str, float], pd.DataFrame]:
    # Time full evaluation wall-clock (LoRA fusion + CudaGraphs shared-state inference + respondent-level OLS)
    torch.cuda.synchronize()
    t_eval_wall0 = time.perf_counter()

    was_training = model.training
    model.eval()
    free_gpu()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(model, eval_dtype=torch.bfloat16):
            preds, stats = predict_probs_grouped(model, eval_bundle, show_progress=True, return_stats=True)
    finally:
        # Always restore training mode and release temporary graph buffers before resuming Grummon 2023 training
        if was_training:
            model.train()
        free_gpu()

    if model.training != was_training:
        raise AssertionError(f"Expected model.training={was_training} after validation, got {model.training}")
    if any((not p.requires_grad) or p.numel() == 0 for p in model.trainable_parameters()):
        raise AssertionError("Trainable parameters were not cleanly restored after fused_eval_context")

    # Compute 3-option choice KL loss and respondent-averaged healthier-selection (`option 0`) ATEs vs `Positive`
    t_ols0 = time.perf_counter()
    num_tasks = y_eval_3col.shape[0]
    if len(preds) != num_tasks:
        raise ValueError(f"Expected {num_tasks} Grummon 2025 task predictions, got {len(preds)}")
    pred_probs_3col = torch.from_numpy(np.stack(preds, axis=0)).to(dtype=torch.float64)
    if pred_probs_3col.shape != (num_tasks, 3):
        raise ValueError(f"Expected ({num_tasks}, 3) predicted probabilities, got {tuple(pred_probs_3col.shape)}")

    g25_kl_loss = float(
        torch.nn.functional.kl_div(
            pred_probs_3col.clamp_min(1e-12).log(),
            y_eval_3col,
            reduction="batchmean",
        )
    )
    u_model = torch.zeros(A_eval.shape[1], dtype=torch.float64)
    u_model.index_add_(0, rec_case_idx, pred_probs_3col[:, 0] / 6.0)
    model_eval_betas = fit_ols_betas(A_eval, u_model)
    beta_diff = model_eval_betas - human_eval_betas
    eval_ols_ms = (time.perf_counter() - t_ols0) * 1000.0
    eval_wall_sec = time.perf_counter() - t_eval_wall0

    metrics = {
        "g25_kl_loss": g25_kl_loss,
        "g25_beta_mse": float(beta_diff.pow(2).mean()),
        "g25_beta_mae": float(beta_diff.abs().mean()),
        "g25_beta_mape_pct": float((beta_diff.abs() / human_eval_betas.abs()).mean() * 100.0),
        "spearman_rho": float(spearmanr(model_eval_betas.numpy(), human_eval_betas.numpy()).statistic),
        "eval_prep_sec": float(stats["prep_sec"]),
        "eval_gpu_sec": float(stats["gpu_sec"]),
        "eval_ols_ms": eval_ols_ms,
        "eval_wall_sec": eval_wall_sec,
        "eval_ms_per_task": (eval_wall_sec * 1000.0) / num_tasks,
    }
    beta_df = pd.DataFrame(
        {
            "human_pop_beta": human_eval_betas.numpy(),
            "model_pop_beta": model_eval_betas.numpy(),
            "diff": beta_diff.numpy(),
            "abs_diff": beta_diff.abs().numpy(),
        },
        index=pd.Index(beta_names, name="grummon2025_arm"),
    )
    return metrics, beta_df


# Run Step-0 untrained baseline evaluation on Grummon 2025 before any Grummon 2023 training
step0_val_metrics, step0_beta_df = validate_grummon2025_population_ols(
    kev_model,
    grummon2025_eval_bundle,
    g25_rec_case_idx,
    A_grummon2025_pop,
    y_grummon2025_3col,
    grummon2025_human_betas,
    GRUMMON2025_BETA_NAMES,
)
display(pd.Series(step0_val_metrics, name="step_0_grummon2025_val").round(4).to_frame())
display(step0_beta_df.round(4))

In [ ]:
# Snapshot all untrained LoRA + pointer-head tensors on CPU so every `(sample_size, beta_lambda)` run starts identically
INIT_TRAINABLE_STATE = {
    name: param.detach().cpu().clone()
    for name, param in kev_model.named_parameters()
    if param.requires_grad
}


def reset_kev_trainable_weights(model: KevPointerModel, init_state: dict[str, torch.Tensor]) -> None:
    # Fail fast if trainable parameter names changed before restoring Step-0 weights onto GPU
    current_names = {name for name, param in model.named_parameters() if param.requires_grad}
    if current_names != set(init_state.keys()):
        raise ValueError("Trainable parameter names do not match INIT_TRAINABLE_STATE snapshot")
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.copy_(init_state[name].to(device=param.device, dtype=param.dtype))
                param.grad = None
    model.train()
    free_gpu()


# Fixed GPU standardized calorie vectors (`kcal / sigma_y`) used to compute differentiable expected outcomes
BEV_STD_WEIGHTS_GPU = torch.tensor(BEV_KCAL_VEC / GRUMMON2023_SIGMA_BEV, dtype=torch.float64, device=device)
SNACK_STD_WEIGHTS_GPU = torch.tensor(SNACK_KCAL_VEC / GRUMMON2023_SIGMA_SNACK, dtype=torch.float64, device=device)

# Pre-tokenize all 7,943 complete Grummon 2023 respondents once in `[bev_choice, snack_choice]` order
t_resp_prep0 = time.perf_counter()
grummon2023_respondent_items: list[dict] = []

for r_idx in tqdm(range(len(g23_complete_caseids)), desc="Preparing Grummon 2023 respondents"):
    bev_row = g23_bev_df.iloc[r_idx]
    snack_row = g23_snack_df.iloc[r_idx]
    r_caseid = str(bev_row["CASEID"])
    r_arm = str(bev_row["arm"])
    if str(snack_row["CASEID"]) != r_caseid or str(snack_row["arm"]) != r_arm:
        raise AssertionError(f"Row {r_idx}: CASEID/arm mismatch between bev_choice and snack_choice")

    r_pair_df = pd.DataFrame([bev_row.drop(labels=["kcal"]), snack_row.drop(labels=["kcal"])]).reset_index(drop=True)
    for t_name, t_oh in zip(["bev_choice", "snack_choice"], r_pair_df["one_hot"], strict=True):
        oh_list = [float(v) for v in t_oh]
        if len(oh_list) != 16 or sum(v == 1.0 for v in oh_list) != 1 or sum(v == 0.0 for v in oh_list) != 15:
            raise ValueError(f"{r_caseid} {t_name}: invalid 16-option one-hot vector")

    r_records = build_long_records(
        r_pair_df,
        persona_cols=GRUMMON2023_PERSONA_COLS,
        include_country=False,
        year="grummon2023",
    )
    r_enc, r_targets = kev_model.prepare(make_grouped_view(r_records))
    r_y_onehot = torch.stack([q["target"].to(dtype=torch.float64, device=device) for q in r_targets], dim=0)
    expected_onehot = torch.tensor(list(r_pair_df["one_hot"]), dtype=torch.float64, device=device)
    if r_y_onehot.shape != (2, 16) or not torch.equal(r_y_onehot, expected_onehot):
        raise AssertionError(f"{r_caseid}: Kev prepared 16-option targets do not match one_hot")

    grummon2023_respondent_items.append(
        {
            "caseid": r_caseid,
            "arm": r_arm,
            "num_tasks": 2,
            "enc": r_enc,
            "x_arm": torch.tensor([encode_grummon2023_arm_row(r_arm)], dtype=torch.float64),
            "y_onehot": r_y_onehot,
            "bev_std_y": float(bev_row["kcal"]) / GRUMMON2023_SIGMA_BEV,
            "snack_std_y": float(snack_row["kcal"]) / GRUMMON2023_SIGMA_SNACK,
        }
    )

resp_prep_sec = time.perf_counter() - t_resp_prep0
if len(grummon2023_respondent_items) != 7943:
    raise AssertionError(f"Expected 7,943 prepared Grummon 2023 respondents, got {len(grummon2023_respondent_items)}")

# Registry keyed by `(sample_size, beta_lambda)` storing step/timing logs and Grummon 2025 test trajectories
experiment_runs: dict[tuple[int, float], dict] = {}

display(
    pd.Series(
        {
            "total_respondents": len(grummon2023_respondent_items),
            "tasks_per_respondent": 2,
            "options_per_task": 16,
            "sigma_bev_kcal": round(GRUMMON2023_SIGMA_BEV, 3),
            "sigma_snack_kcal": round(GRUMMON2023_SIGMA_SNACK, 3),
            "snapshot_trainable_tensors": len(INIT_TRAINABLE_STATE),
            "respondent_prep_sec": round(resp_prep_sec, 3),
            "ms_per_respondent_prep": round((resp_prep_sec * 1000.0) / len(grummon2023_respondent_items), 3),
        },
        name="grummon2023_prep_summary",
    ).to_frame()
)

In [ ]:
# Shared epoch hyperparameters; each optimization step draws `sample_size` Grummon 2023 human respondents
EPOCH_SEED = 42
BASE_LR_SAMPLE_1 = 2e-5               # validated 1-respondent/step base LR; scaled by sqrt(sample_size)


def scale_lr_for_sample_size(sample_size: int) -> float:
    # Square-root AdamW LR scaling keeps 1-epoch parameter travel comparable across respondent sample sizes
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    return BASE_LR_SAMPLE_1 * math.sqrt(sample_size)


def split_into_chunks(items: list, chunk_size: int) -> list[list]:
    # Consecutive chunks of at most `chunk_size` respondents in original sample order
    return [items[start : start + chunk_size] for start in range(0, len(items), chunk_size)]


def predict_chunk(chunk_items: list[dict]) -> torch.Tensor:
    # Shared-prefix forward pass for one micro-chunk -> float64 logits of shape (2 * len(chunk_items), 16)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits_per_respondent = kev_model.forward([it["enc"] for it in chunk_items], shared_prefix=True)
    rows = [
        question_logits.to(dtype=torch.float64)
        for respondent_logits in logits_per_respondent
        for question_logits in respondent_logits
    ]
    out = torch.stack(rows, dim=0)
    if out.shape != (2 * len(chunk_items), 16):
        raise ValueError(f"Expected ({2 * len(chunk_items)}, 16) chunk logits, got {tuple(out.shape)}")
    return out


def make_val_row(
    step: int,
    epoch: int,
    respondents_seen: int,
    train_loss: float,
    metrics: dict[str, float],
    lr: float,
    train_elapsed_sec: float,
) -> dict[str, float | int]:
    # Explicit schema: step/epoch counts, train loss, Grummon 2025 test metrics, LR, and eval/train timings
    return {
        "step": step,
        "epoch": epoch,
        "respondents_seen": respondents_seen,
        "train_loss": train_loss,
        "g25_kl_loss": metrics["g25_kl_loss"],
        "g25_beta_mse": metrics["g25_beta_mse"],
        "g25_beta_mae": metrics["g25_beta_mae"],
        "g25_beta_mape_pct": metrics["g25_beta_mape_pct"],
        "spearman_rho": metrics["spearman_rho"],
        "lr": lr,
        "eval_prep_sec": metrics["eval_prep_sec"],
        "eval_gpu_sec": metrics["eval_gpu_sec"],
        "eval_ols_ms": metrics["eval_ols_ms"],
        "eval_wall_sec": metrics["eval_wall_sec"],
        "eval_ms_per_task": metrics["eval_ms_per_task"],
        "train_elapsed_sec": train_elapsed_sec,
    }


def draw_live_plots(fig, axes, step_rows: list[dict], val_rows: list[dict], rolling_steps: int, steps_per_epoch: int) -> None:
    # Redraw Grummon 2023 train loss, Grummon 2025 test KL / beta MSE / MAE / MAPE, and per-step LR
    for ax in axes.flat:
        ax.clear()
    val_steps = [r["step"] for r in val_rows]
    if step_rows:
        train_steps = [r["step"] for r in step_rows]
        train_loss = pd.Series([r["total_loss"] for r in step_rows])
        axes[0, 0].plot(train_steps, train_loss, alpha=0.3, label="per step")
        axes[0, 0].plot(train_steps, train_loss.rolling(rolling_steps, min_periods=1).mean(), label=f"rolling {rolling_steps}")
        axes[0, 0].legend(fontsize=8)
        axes[1, 2].plot(train_steps, [r["lr"] for r in step_rows])
    for ax, key, title in [
        (axes[0, 1], "g25_kl_loss", "Grummon 2025 KL loss"),
        (axes[0, 2], "g25_beta_mse", "Grummon 2025 beta MSE"),
        (axes[1, 0], "g25_beta_mae", "Grummon 2025 beta MAE"),
        (axes[1, 1], "g25_beta_mape_pct", "Grummon 2025 beta MAPE (%)"),
    ]:
        ax.plot(val_steps, [r[key] for r in val_rows], marker="o")
        ax.set_title(title)
    axes[0, 0].set_title("Grummon 2023 train total loss")
    axes[1, 2].set_title("learning rate")
    last_step = step_rows[-1]["step"] if step_rows else 0
    for ax in axes.flat:
        ax.set_xlabel("step")
        for b in range(steps_per_epoch, last_step + 1, steps_per_epoch):
            ax.axvline(b, color="gray", linestyle="--", linewidth=0.8)
    fig.tight_layout()


def run_grummon2023_epoch(
    *,
    sample_size: int,
    beta_lambda: float,
    lr: float,
    num_epochs: int,
    eval_every: int,
    lr_schedule: str,
    weight_decay: float,
    micro_chunk: int,
    gradient_checkpointing: bool,
) -> dict:
    # Fail fast on invalid training arguments
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    if not isinstance(num_epochs, int) or isinstance(num_epochs, bool) or num_epochs < 1:
        raise ValueError(f"num_epochs must be a positive integer, got {num_epochs!r}")
    if not isinstance(eval_every, int) or isinstance(eval_every, bool) or eval_every < 1:
        raise ValueError(f"eval_every must be a positive integer, got {eval_every!r}")
    if lr_schedule not in ("onecycle", "constant"):
        raise ValueError(f"lr_schedule must be 'onecycle' or 'constant', got {lr_schedule!r}")
    if weight_decay < 0.0:
        raise ValueError(f"weight_decay must be non-negative, got {weight_decay}")
    if not isinstance(micro_chunk, int) or isinstance(micro_chunk, bool) or micro_chunk < 1:
        raise ValueError(f"micro_chunk must be a positive integer, got {micro_chunk!r}")
    if not isinstance(gradient_checkpointing, bool):
        raise ValueError(f"gradient_checkpointing must be a bool, got {gradient_checkpointing!r}")
    if beta_lambda < 0.0:
        raise ValueError(f"beta_lambda must be non-negative, got {beta_lambda}")
    if lr <= 0.0:
        raise ValueError(f"lr must be positive, got {lr}")

    t_wall0 = time.perf_counter()

    # Reset weights to exact Step-0 initialization and seed RNGs so respondent sample order matches across runs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    if gradient_checkpointing:
        kev_model.enable_checkpointing()
    else:
        kev_model.net.lm.gradient_checkpointing_disable()

    rng = random.Random(EPOCH_SEED)
    torch.manual_seed(EPOCH_SEED)
    torch.cuda.manual_seed_all(EPOCH_SEED)

    base_items = list(grummon2023_respondent_items)
    total_respondents = len(base_items)
    steps_per_epoch = math.ceil(total_respondents / sample_size)
    total_steps = steps_per_epoch * num_epochs
    postfix_every = max(1, steps_per_epoch // 30)
    rolling_steps = max(1, round(600 / sample_size))

    try:
        trainable_params = list(kev_model.trainable_parameters())
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay, fused=True)
        if lr_schedule == "onecycle" and total_steps > 1:
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=lr,
                total_steps=total_steps,
                pct_start=0.1,
                div_factor=25.0,
                final_div_factor=1e4,
            )
        else:
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1.0)

        step_rows: list[dict] = []
        val_rows: list[dict] = [make_val_row(0, 0, 0, float("nan"), step0_val_metrics, float("nan"), 0.0)]
        val_display = display(pd.DataFrame(val_rows).set_index("step").round(5), display_id=True)
        fig, axes = plt.subplots(2, 3, figsize=(15, 7))
        plt.close(fig)
        draw_live_plots(fig, axes, step_rows, val_rows, rolling_steps, steps_per_epoch)
        plot_display = display(fig, display_id=True)
        final_beta_df = step0_beta_df
        cum_train_sec = 0.0
        cum_eval_sec = 0.0
        respondents_seen = 0

        pbar = tqdm(range(1, total_steps + 1), total=total_steps, desc=f"S={sample_size} | lambda={beta_lambda:g}")
        for step_idx in pbar:
            epoch_idx = (step_idx - 1) // steps_per_epoch + 1
            step_in_epoch = (step_idx - 1) % steps_per_epoch + 1
            if step_in_epoch == 1:
                epoch_items = list(base_items)
                rng.shuffle(epoch_items)
            start_idx = (step_in_epoch - 1) * sample_size
            sample_items = epoch_items[start_idx : start_idx + sample_size]
            n_sample_resp = len(sample_items)
            respondents_seen += n_sample_resp

            torch.cuda.synchronize()
            t_step0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)

            sample_chunks = split_into_chunks(sample_items, micro_chunk)

            # 1. Pass 1: forward every micro-chunk without an autograd graph; record per-chunk CUDA RNG state
            chunk_rng_states = []
            chunk_logits_list = []
            with torch.no_grad():
                for chunk in sample_chunks:
                    chunk_rng_states.append(torch.cuda.get_rng_state())
                    chunk_logits_list.append(predict_chunk(chunk))
            task_logits = torch.cat(chunk_logits_list, dim=0)
            torch.cuda.synchronize()
            t_after_fwd = time.perf_counter()

            # 2. Full-sample KL + standardized Grummon 2023 calorie OLS loss on detached leaf `task_logits`
            task_logits.requires_grad_(True)
            X_arm_sample = torch.cat([it["x_arm"] for it in sample_items], dim=0)
            y_onehot_sample = torch.cat([it["y_onehot"] for it in sample_items], dim=0)
            bev_std_sample = torch.tensor([it["bev_std_y"] for it in sample_items], dtype=torch.float64, device=device)
            snack_std_sample = torch.tensor([it["snack_std_y"] for it in sample_items], dtype=torch.float64, device=device)

            if X_arm_sample.shape[0] >= X_arm_sample.shape[1]:
                svals = torch.linalg.svdvals(X_arm_sample)
                has_ols = float(svals[-1]) > OLS_SVD_RCOND * float(svals[0])
            else:
                has_ols = False

            if has_ols:
                A_arm_sample = torch.linalg.pinv(X_arm_sample)[1:].to(device=device)
                total_loss, kl_loss, beta_loss = compute_grummon2023_loss(
                    task_logits,
                    y_onehot_sample,
                    bev_std_sample,
                    snack_std_sample,
                    A_arm_sample,
                    BEV_STD_WEIGHTS_GPU,
                    SNACK_STD_WEIGHTS_GPU,
                    beta_lambda=beta_lambda,
                )
                raw_beta_val = float(beta_loss.detach().cpu())
            else:
                A_arm_sample = None
                beta_loss = None
                log_probs = torch.log_softmax(task_logits, dim=-1)
                kl_loss = torch.nn.functional.kl_div(log_probs, y_onehot_sample, reduction="batchmean")
                total_loss = kl_loss
                raw_beta_val = float("nan")

            kl_val = float(kl_loss.detach().cpu())
            total_val = float(total_loss.detach().cpu())
            total_loss.backward()
            questions_per_chunk = [len(chunk_logits) for chunk_logits in chunk_logits_list]
            grad_per_chunk = task_logits.grad.split(questions_per_chunk, dim=0)
            torch.cuda.synchronize()
            t_after_loss = time.perf_counter()

            # 3. Pass 2: replay each micro-chunk with exact dropout RNG state and accumulate weight gradients
            for chunk, rng_state, chunk_grad in zip(sample_chunks, chunk_rng_states, grad_per_chunk, strict=True):
                torch.cuda.set_rng_state(rng_state)
                chunk_logits = predict_chunk(chunk)
                chunk_logits.backward(chunk_grad)
                del chunk_logits

            # 4. Clip gradients, apply AdamW update, and advance LR schedule
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            if not bool(torch.isfinite(grad_norm)):
                raise FloatingPointError(f"Non-finite gradient norm {float(grad_norm)} at step {step_idx}")
            grad_norm_val = float(grad_norm.detach().cpu())
            step_lr = float(optimizer.param_groups[0]["lr"])
            optimizer.step()
            if step_idx < total_steps:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            del (
                chunk_logits_list,
                task_logits,
                grad_per_chunk,
                X_arm_sample,
                y_onehot_sample,
                bev_std_sample,
                snack_std_sample,
                A_arm_sample,
                total_loss,
                kl_loss,
                beta_loss,
            )
            torch.cuda.synchronize()
            t_after_bwd = time.perf_counter()

            fwd_ms = (t_after_fwd - t_step0) * 1000.0
            loss_ms = (t_after_loss - t_after_fwd) * 1000.0
            bwd_opt_ms = (t_after_bwd - t_after_loss) * 1000.0
            step_ms = (t_after_bwd - t_step0) * 1000.0
            resp_ms = step_ms / n_sample_resp
            cum_train_sec += t_after_bwd - t_step0

            step_rows.append(
                {
                    "step": step_idx,
                    "epoch": epoch_idx,
                    "num_respondents": n_sample_resp,
                    "respondents_seen": respondents_seen,
                    "has_ols": has_ols,
                    "kl_loss": kl_val,
                    "raw_beta_loss": raw_beta_val,
                    "weighted_beta_loss": beta_lambda * raw_beta_val if has_ols else 0.0,
                    "total_loss": total_val,
                    "grad_norm": grad_norm_val,
                    "lr": step_lr,
                    "fwd_ms": fwd_ms,
                    "loss_ms": loss_ms,
                    "bwd_opt_ms": bwd_opt_ms,
                    "step_ms": step_ms,
                    "ms_per_respondent": resp_ms,
                }
            )

            if step_idx % postfix_every == 0 or step_idx == total_steps:
                recent = step_rows[-rolling_steps:]
                recent_betas = [r["raw_beta_loss"] for r in recent if r["has_ols"]]
                pbar.set_postfix(
                    kl=f"{float(np.mean([r['kl_loss'] for r in recent])):.3f}",
                    beta=f"{float(np.mean(recent_betas)):.4f}" if recent_betas else "n/a",
                    step_ms=f"{float(np.mean([r['step_ms'] for r in recent])):.1f}",
                    resp_ms=f"{float(np.mean([r['ms_per_respondent'] for r in recent])):.1f}",
                )

            # 5. Evaluate held-out Grummon 2025 population OLS every `eval_every` steps and at the final step
            if step_idx % eval_every == 0 or step_idx == total_steps:
                v_metrics, final_beta_df = validate_grummon2025_population_ols(
                    kev_model,
                    grummon2025_eval_bundle,
                    g25_rec_case_idx,
                    A_grummon2025_pop,
                    y_grummon2025_3col,
                    grummon2025_human_betas,
                    GRUMMON2025_BETA_NAMES,
                )
                cum_eval_sec += v_metrics["eval_wall_sec"]
                previous_eval_step = val_rows[-1]["step"]
                steps_since_previous_eval = [r for r in step_rows if r["step"] > previous_eval_step]
                interval_train_loss = float(np.mean([r["total_loss"] for r in steps_since_previous_eval]))
                val_rows.append(
                    make_val_row(step_idx, epoch_idx, respondents_seen, interval_train_loss, v_metrics, step_lr, cum_train_sec)
                )
                val_display.update(pd.DataFrame(val_rows).set_index("step").round(5))

            if step_idx % postfix_every == 0 or step_idx % eval_every == 0 or step_idx == total_steps:
                draw_live_plots(fig, axes, step_rows, val_rows, rolling_steps, steps_per_epoch)
                plot_display.update(fig)
    finally:
        for param in kev_model.trainable_parameters():
            param.grad = None
        optimizer = scheduler = trainable_params = None
        chunk_logits_list = chunk_logits = task_logits = grad_per_chunk = None
        X_arm_sample = y_onehot_sample = bev_std_sample = snack_std_sample = A_arm_sample = None
        total_loss = kl_loss = beta_loss = grad_norm = None
        free_gpu()

    total_wall_sec = time.perf_counter() - t_wall0
    step_df = pd.DataFrame(step_rows)
    val_history_df = pd.DataFrame(val_rows).set_index("step")
    ols_steps = step_df[step_df["has_ols"]]
    edge_window = max(1, round(0.15 * len(step_df)))
    if len(ols_steps) > 0:
        ols_edge_window = max(1, round(0.15 * len(ols_steps)))
        raw_beta_first15 = float(ols_steps["raw_beta_loss"].iloc[:ols_edge_window].mean())
        raw_beta_last15 = float(ols_steps["raw_beta_loss"].iloc[-ols_edge_window:].mean())
        weighted_beta_last15 = float(ols_steps["weighted_beta_loss"].iloc[-ols_edge_window:].mean())
    else:
        raw_beta_first15 = float("nan")
        raw_beta_last15 = float("nan")
        weighted_beta_last15 = float("nan")

    summary = pd.Series(
        {
            "sample_size": sample_size,
            "beta_lambda": beta_lambda,
            "lr": lr,
            "num_epochs": num_epochs,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": total_steps,
            "eval_every": eval_every,
            "lr_schedule": lr_schedule,
            "weight_decay": weight_decay,
            "micro_chunk": micro_chunk,
            "gradient_checkpointing": gradient_checkpointing,
            "rank_deficient_steps": int((~step_df["has_ols"]).sum()),
            "train_kl_first15pct": float(step_df["kl_loss"].iloc[:edge_window].mean()),
            "train_kl_last15pct": float(step_df["kl_loss"].iloc[-edge_window:].mean()),
            "train_raw_beta_first15pct": raw_beta_first15,
            "train_raw_beta_last15pct": raw_beta_last15,
            "train_weighted_beta_last15pct": weighted_beta_last15,
            "train_total_first15pct": float(step_df["total_loss"].iloc[:edge_window].mean()),
            "train_total_last15pct": float(step_df["total_loss"].iloc[-edge_window:].mean()),
            "mean_grad_norm": float(step_df["grad_norm"].mean()),
            "mean_fwd_ms": float(step_df["fwd_ms"].mean()),
            "mean_loss_ms": float(step_df["loss_ms"].mean()),
            "mean_bwd_opt_ms": float(step_df["bwd_opt_ms"].mean()),
            "mean_step_ms": float(step_df["step_ms"].mean()),
            "mean_ms_per_respondent": (cum_train_sec * 1000.0) / respondents_seen,
            "mean_eval_wall_sec": float(val_history_df["eval_wall_sec"].iloc[1:].mean()),
            "mean_eval_gpu_sec": float(val_history_df["eval_gpu_sec"].iloc[1:].mean()),
            "train_only_sec": cum_train_sec,
            "eval_only_sec": cum_eval_sec,
            "total_wall_sec": total_wall_sec,
        },
        name=f"S_{sample_size}_lambda_{beta_lambda:g}_summary",
    )
    val_display.update(val_history_df.round(5))

    return {
        "sample_size": sample_size,
        "beta_lambda": beta_lambda,
        "lr": lr,
        "num_epochs": num_epochs,
        "eval_every": eval_every,
        "lr_schedule": lr_schedule,
        "weight_decay": weight_decay,
        "micro_chunk": micro_chunk,
        "gradient_checkpointing": gradient_checkpointing,
        "summary": summary,
        "step_df": step_df,
        "val_history_df": val_history_df,
        "final_beta_df": final_beta_df,
    }


print("Defined scale_lr_for_sample_size(sample_size) and run_grummon2023_epoch(...)")

# Test micro-batching

In [ ]:
# Gradient check: verify two-pass micro-chunking (Method B) matches one full-sample backward (Method A) on Grummon 2023
CHECK_PER_ARM = 3         # 3 respondents x 5 FOPL arms = 15 respondents (full 5/5 arm rank, fits in single graph)
CHUNK = 3                 # 5 micro-chunks of 3 respondents
LAM = 5.0                 # beta_lambda > 0 so both 16-option KL and standardized calorie OLS gradients are tested

reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
kev_model.enable_checkpointing()
params = list(kev_model.trainable_parameters())

# Pick `CHECK_PER_ARM` respondents from each of the 5 Grummon 2023 arms so `X_arm_check` is full rank
sample: list[dict] = []
for arm_name in GRUMMON2023_ARM_ORDER:
    arm_matches = [it for it in grummon2023_respondent_items if it["arm"] == arm_name][:CHECK_PER_ARM]
    if len(arm_matches) != CHECK_PER_ARM:
        raise ValueError(f"Expected at least {CHECK_PER_ARM} respondents in arm {arm_name!r}")
    sample.extend(arm_matches)

chunks = split_into_chunks(sample, CHUNK)
X_arm_check = torch.cat([resp["x_arm"] for resp in sample], dim=0)
y_onehot_check = torch.cat([resp["y_onehot"] for resp in sample], dim=0)
bev_std_check = torch.tensor([resp["bev_std_y"] for resp in sample], dtype=torch.float64, device=device)
snack_std_check = torch.tensor([resp["snack_std_y"] for resp in sample], dtype=torch.float64, device=device)
A_arm_check = build_ols_projector(X_arm_check).to(device=device)


def collect_weight_gradients() -> torch.Tensor:
    # Flatten every trainable parameter gradient into one float32 vector, then clear `.grad`
    flat = torch.cat([p.grad.flatten().float() for p in params])
    for p in params:
        p.grad = None
    return flat


def method_b() -> tuple[float, torch.Tensor, list[torch.Tensor]]:
    # Two-pass micro-chunked forward/backward capturing per-chunk CUDA RNG states exactly as training does
    saved_states: list[torch.Tensor] = []
    logits_parts: list[torch.Tensor] = []
    with torch.no_grad():
        for chunk in chunks:
            saved_states.append(torch.cuda.get_rng_state())
            logits_parts.append(predict_chunk(chunk))
    all_logits = torch.cat(logits_parts, dim=0)

    all_logits.requires_grad_(True)
    loss, _, _ = compute_grummon2023_loss(
        all_logits,
        y_onehot_check,
        bev_std_check,
        snack_std_check,
        A_arm_check,
        BEV_STD_WEIGHTS_GPU,
        SNACK_STD_WEIGHTS_GPU,
        beta_lambda=LAM,
    )
    loss.backward()
    questions_per_chunk = [len(part) for part in logits_parts]
    grad_per_chunk = all_logits.grad.split(questions_per_chunk, dim=0)

    for chunk, rng_state, chunk_grad in zip(chunks, saved_states, grad_per_chunk, strict=True):
        torch.cuda.set_rng_state(rng_state)
        chunk_logits = predict_chunk(chunk)
        chunk_logits.backward(chunk_grad)
    return float(loss.detach()), collect_weight_gradients(), saved_states


def method_a(saved_states: list[torch.Tensor]) -> tuple[float, torch.Tensor]:
    # Single full-sample autograd graph replaying the exact per-chunk dropout RNG states from `method_b`
    logits_parts: list[torch.Tensor] = []
    for chunk, rng_state in zip(chunks, saved_states, strict=True):
        torch.cuda.set_rng_state(rng_state)
        logits_parts.append(predict_chunk(chunk))
    all_logits = torch.cat(logits_parts, dim=0)
    loss, _, _ = compute_grummon2023_loss(
        all_logits,
        y_onehot_check,
        bev_std_check,
        snack_std_check,
        A_arm_check,
        BEV_STD_WEIGHTS_GPU,
        SNACK_STD_WEIGHTS_GPU,
        beta_lambda=LAM,
    )
    loss.backward()
    return float(loss.detach()), collect_weight_gradients()


try:
    loss_b, grad_b, saved_rng_states = method_b()
    loss_a1, grad_a1 = method_a(saved_rng_states)
    loss_a2, grad_a2 = method_a(saved_rng_states)

    rel_a1_a2 = float((grad_a1 - grad_a2).norm() / grad_a1.norm())
    cos_a1_a2 = float(torch.nn.functional.cosine_similarity(grad_a1, grad_a2, dim=0))
    rel_a1_b = float((grad_a1 - grad_b).norm() / grad_a1.norm())
    cos_a1_b = float(torch.nn.functional.cosine_similarity(grad_a1, grad_b, dim=0))

    print(f"sample={len(sample)} respondents ({len(sample) * 2} 16-option tasks) | chunk={CHUNK} | lambda={LAM:g}")
    print(f"loss  A={loss_a1:.8f}  A again={loss_a2:.8f}  B={loss_b:.8f}")
    print(f"A vs A again (noise floor): rel={rel_a1_a2:.2e}  cos={cos_a1_a2:.8f}")
    print(f"A vs B       (two-pass):    rel={rel_a1_b:.2e}  cos={cos_a1_b:.8f}")
finally:
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)

# Find max micro chunk

In [ ]:
# Settings for the worst-case Grummon 2023 16-option micro-chunk VRAM search
SEARCH_GRADIENT_CHECKPOINTING = False
SEARCH_MAX_CHUNK = 64


def respondent_token_count(item: dict) -> int:
    # Total packed tokens in one respondent's encoding (persona prefix + 16-bev branch + 16-snack branch)
    return len(item["enc"]["ids"])


respondents_by_size = sorted(grummon2023_respondent_items, key=respondent_token_count, reverse=True)

largest_respondents_df = pd.DataFrame(
    {
        "caseid": [it["caseid"] for it in respondents_by_size[:10]],
        "arm": [it["arm"] for it in respondents_by_size[:10]],
        "num_tasks": [it["num_tasks"] for it in respondents_by_size[:10]],
        "tokens": [respondent_token_count(it) for it in respondents_by_size[:10]],
    }
)
display(largest_respondents_df)

In [ ]:
def chunk_fits(chunk_size: int) -> tuple[bool, float]:
    # Test one Pass-2 forward + backward + fused AdamW step on the `chunk_size` longest Grummon 2023 respondents
    worst_chunk = respondents_by_size[:chunk_size]
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    torch.cuda.reset_peak_memory_stats()

    optimizer = torch.optim.AdamW(list(kev_model.trainable_parameters()), lr=1e-8, fused=True)
    chunk_logits = None
    fits = True
    try:
        chunk_logits = predict_chunk(worst_chunk)
        chunk_logits.sum().backward()
        optimizer.step()
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        fits = False
    finally:
        chunk_logits = None
        optimizer = None
        free_gpu()

    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    return fits, peak_gib


if SEARCH_GRADIENT_CHECKPOINTING:
    kev_model.enable_checkpointing()
else:
    kev_model.net.lm.gradient_checkpointing_disable()

total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
search_rows: list[dict] = []
largest_fit = 0
smallest_fail = None


def record_trial(chunk_size: int, fits: bool, peak_gib: float) -> None:
    search_rows.append(
        {
            "chunk_size": chunk_size,
            "tokens": sum(respondent_token_count(it) for it in respondents_by_size[:chunk_size]),
            "fits": fits,
            "peak_gib": round(peak_gib, 2) if fits else float("nan"),
            "peak_pct_of_vram": round(100.0 * peak_gib / total_vram_gib, 1) if fits else float("nan"),
        }
    )


# Phase 1: double chunk_size (1, 2, 4, ...) until OOM or `SEARCH_MAX_CHUNK`
chunk_size = 1
while chunk_size <= SEARCH_MAX_CHUNK:
    fits, peak_gib = chunk_fits(chunk_size)
    record_trial(chunk_size, fits, peak_gib)
    if not fits:
        smallest_fail = chunk_size
        break
    largest_fit = chunk_size
    chunk_size *= 2

# Phase 2: binary search between `largest_fit` and `smallest_fail`
if smallest_fail is not None:
    low = largest_fit
    high = smallest_fail
    while high - low > 1:
        middle = (low + high) // 2
        fits, peak_gib = chunk_fits(middle)
        record_trial(middle, fits, peak_gib)
        if fits:
            low = middle
        else:
            high = middle
    largest_fit = low

if largest_fit < 1:
    raise RuntimeError("Even chunk_size=1 ran out of VRAM; enable SEARCH_GRADIENT_CHECKPOINTING=True")

MAX_FITTING_MICRO_CHUNK = largest_fit
display(pd.DataFrame(search_rows).sort_values("chunk_size").set_index("chunk_size"))
print(
    f"gradient_checkpointing={SEARCH_GRADIENT_CHECKPOINTING} | total VRAM {total_vram_gib:.1f} GiB | "
    f"MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK}"
    + ("" if smallest_fail is not None else f" (no OOM up to SEARCH_MAX_CHUNK={SEARCH_MAX_CHUNK})")
)

# Overfit first

In [ ]:
# Overfit sanity check: multiple passes over Grummon 2023 with pure KL/CE, constant LR, and zero weight decay
OVERFIT_SAMPLE_SIZE = 128
OVERFIT_NUM_EPOCHS = 10
OVERFIT_LR = scale_lr_for_sample_size(OVERFIT_SAMPLE_SIZE)
OVERFIT_EVAL_EVERY = math.ceil(len(grummon2023_respondent_items) / OVERFIT_SAMPLE_SIZE)  # 1 Grummon 2025 eval per epoch
OVERFIT_MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
OVERFIT_GRADIENT_CHECKPOINTING = SEARCH_GRADIENT_CHECKPOINTING

overfit_run = run_grummon2023_epoch(
    sample_size=OVERFIT_SAMPLE_SIZE,
    beta_lambda=0.0,
    lr=OVERFIT_LR,
    num_epochs=OVERFIT_NUM_EPOCHS,
    eval_every=OVERFIT_EVAL_EVERY,
    lr_schedule="constant",
    weight_decay=0.0,
    micro_chunk=OVERFIT_MICRO_CHUNK,
    gradient_checkpointing=OVERFIT_GRADIENT_CHECKPOINTING,
)

# Train

In [ ]:
# Respondent sample size per step (`SAMPLE_SIZE = 128` -> ~25.6 respondents/arm across 5 arms, 0% rank deficiency)
SAMPLE_SIZE = 128
# Number of full passes over all 7,943 Grummon 2023 respondents per run
NUM_EPOCHS = 5
# Steps between held-out Grummon 2025 evaluations (`31` = twice per 63-step epoch, plus final step)
EVAL_EVERY = 31
# Learning-rate schedule ("onecycle" or "constant") and AdamW weight decay
LR_SCHEDULE = "onecycle"
WEIGHT_DECAY = 0.01
# Wire micro-chunk size and checkpointing directly from the worst-case VRAM search above
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
GRADIENT_CHECKPOINTING = SEARCH_GRADIENT_CHECKPOINTING
SAMPLE_LR = scale_lr_for_sample_size(SAMPLE_SIZE)
steps_per_epoch = math.ceil(len(grummon2023_respondent_items) / SAMPLE_SIZE)

display(
    pd.Series(
        {
            "sample_size": SAMPLE_SIZE,
            "mean_respondents_per_arm": round(SAMPLE_SIZE / 5.0, 2),
            "num_epochs": NUM_EPOCHS,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": steps_per_epoch * NUM_EPOCHS,
            "eval_every": EVAL_EVERY,
            "lr_schedule": LR_SCHEDULE,
            "weight_decay": WEIGHT_DECAY,
            "scaled_max_lr": round(SAMPLE_LR, 7),
            "micro_chunk": MICRO_CHUNK,
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        },
        name="grummon2023_train_config",
    ).to_frame()
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with pure 16-option choice KL/CE baseline (`beta_lambda = 0.0`)
beta_lambda = 0.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with mild standardized calorie OLS beta weight (`beta_lambda = 1.0`)
beta_lambda = 1.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with standardized calorie OLS beta weight (`beta_lambda = 2.0`)
beta_lambda = 2.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with standardized calorie OLS beta weight (`beta_lambda = 3.0`)
beta_lambda = 3.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with standardized calorie OLS beta weight (`beta_lambda = 4.0`)
beta_lambda = 4.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with moderate standardized calorie OLS beta weight (`beta_lambda = 5.0`)
beta_lambda = 5.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over Grummon 2023 at `SAMPLE_SIZE` respondents/step with strong standardized calorie OLS beta weight (`beta_lambda = 20.0`)
beta_lambda = 20.0
experiment_runs[(SAMPLE_SIZE, beta_lambda)] = run_grummon2023_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=beta_lambda,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Compare Grummon 2023 train metrics, held-out Grummon 2025 test OLS/KL metrics, and timings across completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_grummon2023_epoch` cell first so `experiment_runs` is populated")
ran_keys = sorted(experiment_runs.keys())

compare_rows = [
    {
        "run": "step_0 (untrained)",
        "sample_size": 0,
        "beta_lambda": float("nan"),
        "num_epochs": 0,
        "steps_per_epoch": 0,
        "train_kl_last15pct": float("nan"),
        "train_raw_beta_last15pct": float("nan"),
        "g25_kl_loss": step0_val_metrics["g25_kl_loss"],
        "g25_beta_mse": step0_val_metrics["g25_beta_mse"],
        "g25_beta_mae": step0_val_metrics["g25_beta_mae"],
        "g25_beta_mape_pct": step0_val_metrics["g25_beta_mape_pct"],
        "spearman_rho": step0_val_metrics["spearman_rho"],
        "mean_step_ms": float("nan"),
        "mean_ms_per_respondent": float("nan"),
        "mean_eval_wall_sec": step0_val_metrics["eval_wall_sec"],
        "train_only_sec": 0.0,
        "eval_only_sec": step0_val_metrics["eval_wall_sec"],
        "total_wall_sec": step0_val_metrics["eval_wall_sec"],
    }
]
for s_size, lam in ran_keys:
    run = experiment_runs[(s_size, lam)]
    s = run["summary"]
    final_val = run["val_history_df"].iloc[-1]
    compare_rows.append(
        {
            "run": f"S={s_size}, lambda={lam:g}",
            "sample_size": s_size,
            "beta_lambda": lam,
            "num_epochs": int(s["num_epochs"]),
            "steps_per_epoch": int(s["steps_per_epoch"]),
            "train_kl_last15pct": float(s["train_kl_last15pct"]),
            "train_raw_beta_last15pct": float(s["train_raw_beta_last15pct"]),
            "g25_kl_loss": float(final_val["g25_kl_loss"]),
            "g25_beta_mse": float(final_val["g25_beta_mse"]),
            "g25_beta_mae": float(final_val["g25_beta_mae"]),
            "g25_beta_mape_pct": float(final_val["g25_beta_mape_pct"]),
            "spearman_rho": float(final_val["spearman_rho"]),
            "mean_step_ms": float(s["mean_step_ms"]),
            "mean_ms_per_respondent": float(s["mean_ms_per_respondent"]),
            "mean_eval_wall_sec": float(s["mean_eval_wall_sec"]),
            "train_only_sec": float(s["train_only_sec"]),
            "eval_only_sec": float(s["eval_only_sec"]),
            "total_wall_sec": float(s["total_wall_sec"]),
        }
    )

comparison_df = pd.DataFrame(compare_rows).set_index("run")
display(comparison_df.round(5))

# Compare the 5 predicted Grummon 2025 FOPL arm ATEs against ground-truth human RCT ATEs
grummon2025_betas_compare_df = pd.DataFrame(
    {
        "human_pop_beta": step0_beta_df["human_pop_beta"],
        "step_0": step0_beta_df["model_pop_beta"],
        **{
            f"S={s_size}, lambda={lam:g}": experiment_runs[(s_size, lam)]["final_beta_df"]["model_pop_beta"]
            for s_size, lam in ran_keys
        },
    }
)
display(grummon2025_betas_compare_df.round(4))

In [ ]:
# Decompose the 5-term Grummon 2025 `g25_beta_mse` (`(1/5) * sum_k e_k^2`) arm-by-arm across completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_grummon2023_epoch` cell first so `experiment_runs` is populated")

decomp_human = step0_beta_df["human_pop_beta"]
decomp_cols: dict[str, pd.Series] = {"human_beta": decomp_human}
sq_err_by_run: dict[str, pd.Series] = {}

for s_size, lam in sorted(experiment_runs.keys()):
    col_tag = f"lam_{lam:g}"
    pred_s = experiment_runs[(s_size, lam)]["final_beta_df"]["model_pop_beta"]
    if not pred_s.index.equals(decomp_human.index):
        raise ValueError(f"Beta index mismatch for run {(s_size, lam)}")
    decomp_cols[f"pred_{col_tag}"] = pred_s
    sq_err_by_run[col_tag] = (pred_s - decomp_human) ** 2

for col_tag, sq_s in sq_err_by_run.items():
    decomp_cols[f"sq_err_{col_tag}"] = sq_s

grummon2025_mse_decomposition_df = pd.DataFrame(decomp_cols)
mean_row = pd.DataFrame(
    [{col: float(grummon2025_mse_decomposition_df[col].mean()) for col in grummon2025_mse_decomposition_df.columns}],
    index=pd.Index(["MEAN (= g25_beta_mse on sq_err cols)"], name="grummon2025_arm"),
)
display(pd.concat([grummon2025_mse_decomposition_df, mean_row]).round(5))